In [ ]:
# Latest part's summary figure (Part 5). Every number on it is measured again further down.
import glob, os, warnings
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.patches import FancyBboxPatch

SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, RING = "#e1e0d9", "#c3c2b7", "#e6e5df"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
SEQ = ["#f0f6fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
GOOD = "#006300"


def _style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
        "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.facecolor": SURFACE,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
        "axes.grid": False, "figure.facecolor": PAGE, "savefig.facecolor": PAGE,
    })


def _title(ax, title, sub):
    ax.text(0, 1.115, title, transform=ax.transAxes, fontsize=12.5, fontweight="bold", color=INK, va="bottom")
    ax.text(0, 1.035, sub, transform=ax.transAxes, fontsize=9.5, color=INK2, va="bottom")


def _tile(fig, x, y, w, h, label, value, note=None):
    ax = fig.add_axes([x, y, w, h])
    ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.06",
                                transform=ax.transAxes, facecolor=SURFACE, edgecolor=RING, lw=1))
    ax.text(0.06, 0.70, label, transform=ax.transAxes, fontsize=10, color=INK2, va="center")
    ax.text(0.06, 0.40, value, transform=ax.transAxes, fontsize=22, fontweight="bold", color=INK, va="center")
    if note:
        ax.text(0.06, 0.13, note, transform=ax.transAxes, fontsize=8.8,
                color=GOOD if note.startswith("+") else MUTED, va="center")


def _heat(fig, rect, heat):
    ax = fig.add_axes(rect)
    rate = np.asarray(heat["rate"], dtype=float)
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    im = ax.imshow(rate, origin="lower", cmap=cmap, norm=PowerNorm(0.45, vmin=0, vmax=rate.max()), aspect="auto")
    n = rate.shape[0]
    ax.set_xticks([0, n - 1], ["low", "high"])
    ax.set_yticks([0, n - 1], ["low", "high"])
    ax.set_xlabel(f"{heat['xname']}  (deciles)  →")
    ax.set_ylabel(f"{heat['yname']}  (deciles)  →")
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
    for (i, j) in [np.unravel_index(rate.argmax(), rate.shape), np.unravel_index(rate.argmin(), rate.shape)]:
        v = rate[i, j]
        ax.text(j, i, f"{v:.1%}" if v >= 0.001 else "<0.1%", ha="center", va="center", fontsize=8.5,
                fontweight="bold", color="#ffffff" if v > rate.max() * 0.3 else INK)
    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, colors=MUTED, length=0)
    cb.set_ticks([t for t in (0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30) if t <= rate.max()])
    cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    _title(ax, "Where the fraud is", f"fraud rate per cell · base rate {heat['base']:.2%}")


def _dumbbell(fig, rect, te_gain):
    """te_gain: list of (column, raw single-feature AUC, target-encoded AUC), top first."""
    ax = fig.add_axes(rect)
    cols = [c for c, _, _ in te_gain][::-1]
    raw = np.array([r for _, r, _ in te_gain][::-1])
    enc = np.array([e for _, _, e in te_gain][::-1])
    yy = np.arange(len(cols))
    ax.hlines(yy, raw, enc, color=AXIS, lw=2, zorder=1)
    ax.scatter(raw, yy, s=52, color=MUTED, edgecolor=SURFACE, linewidth=2, zorder=2, label="raw value")
    ax.scatter(enc, yy, s=52, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3, label="target-encoded value")
    for yv, r, e in zip(yy, raw, enc):
        ax.text(max(r, e) + 0.004, yv, f"+{(e - r) * 100:.1f}", va="center", fontsize=8.5, color=INK2)
    ax.set_yticks(yy, cols)
    ax.tick_params(axis="y", length=0, labelsize=9, labelcolor=INK)
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(min(raw.min(), enc.min()) - 0.01, max(raw.max(), enc.max()) + 0.03)
    ax.set_xlabel("single-feature AUC (fold 0)")
    ax.legend(loc="upper right", frameon=False, fontsize=8.5, handletextpad=0.2)
    _title(ax, "Every value has its own risk", "per-value target encoding vs the raw number (AUC × 100 gain)")


def _depth(fig, rect, depth):
    """depth: dict(leaves=[...], series={label: [auc per leaves or None]})"""
    ax = fig.add_axes(rect)
    lv = np.asarray(depth["leaves"])
    colors = [MUTED, BLUE]
    for (lab, au), col in zip(depth["series"].items(), colors):
        au = np.array([np.nan if v is None else v for v in au], dtype=float)
        ok = ~np.isnan(au)
        ax.plot(lv[ok], au[ok], color=col, lw=2, solid_capstyle="round", zorder=2)
        ax.scatter(lv[ok], au[ok], s=46, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        last = np.where(ok)[0][-1]
        ax.text(lv[last] * 1.12, au[last], lab, va="center", fontsize=9, color=INK2)
        k = int(np.nanargmax(au))
        ax.annotate(f"{au[k]:.4f}", (lv[k], au[k]), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold", color=INK)
    ax.set_xscale("log", base=2)
    ax.set_xticks(lv, [str(v) for v in lv])
    ax.minorticks_off()
    ax.set_xlim(lv.min() / 1.3, lv.max() * 2.6)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlabel("LightGBM num_leaves (log scale)")
    ax.set_ylabel("fold-0 AUC")
    lo, hi = np.nanmin([v for a in depth["series"].values() for v in a if v is not None]), \
        np.nanmax([v for a in depth["series"].values() for v in a if v is not None])
    ax.set_ylim(lo - (hi - lo) * 0.08, hi + (hi - lo) * 0.16)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, "Shallow trees win", "the signal is close to additive, like a logistic model")


def _ladder(fig, rect, ladder):
    ax = fig.add_axes(rect)
    steps = [s for s, _ in ladder]
    cv = np.array([v for _, v in ladder])
    yy = np.arange(len(steps))[::-1]
    gain = (cv - cv[0]) * 1e4
    colors = [AXIS] + [AQUA] * (len(cv) - 2) + [ORANGE]
    ax.barh(yy, np.maximum(gain, 0.15), height=0.5, color=colors, zorder=2)
    for yv, g, v in zip(yy, gain, cv):
        ax.text(max(g, 0) + gain.max() * 0.02, yv, f"{v:.5f}" + (f"   +{g:.1f}" if g > 0 else "   baseline"),
                va="center", fontsize=9.5, color=INK)
    ax.set_yticks(yy, steps)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(0, gain.max() * 1.45)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("5-fold CV AUC gain over the baseline (× 1e-4)")
    _title(ax, "What moved the score", "5-fold out-of-fold AUC, the same folds for every model")


def _blend(fig, rect, blend):
    ax = fig.add_axes(rect)
    ax.set_axis_off()
    cols = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]
    bottom = 1.0
    for (nm, w), c in zip(blend, cols):
        ax.add_patch(FancyBboxPatch((0.0, bottom - w + 0.006), 0.30, w - 0.012,
                                    boxstyle="round,pad=0,rounding_size=0.012", transform=ax.transAxes,
                                    facecolor=c, edgecolor="none"))
        if w >= 0.06:
            ax.text(0.03, bottom - w / 2, f"{w:.0%}", transform=ax.transAxes, va="center", fontsize=10,
                    fontweight="bold", color=INK if c == YELLOW else "#ffffff")
        ax.text(0.36, bottom - w / 2, nm, transform=ax.transAxes, va="center", fontsize=9.5, color=INK)
        bottom -= w
    _title(ax, "The stack", "share of the stacker's logit weights")


def draw(stats, heat, te_gain, depth, ladder, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _heat(fig, [0.035, 0.425, 0.24, 0.29], heat)
    _dumbbell(fig, [0.43, 0.425, 0.22, 0.29], te_gain)
    _depth(fig, [0.72, 0.425, 0.20, 0.29], depth)
    _ladder(fig, [0.20, 0.05, 0.47, 0.26], ladder)
    _blend(fig, [0.74, 0.05, 0.22, 0.26], blend)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------
# Part 2 panels: two views of the data (target-encoded vs raw), what each candidate adds,
# smoothing as an extra view, the Part 1 -> Part 2 slope, the hill-climb weights.

VIEW_COLORS = {"target-encoded view": BLUE, "raw view": ORANGE, "neural net": AQUA, "joint GLM": YELLOW}


def _corr(fig, rect, names, corr):
    ax = fig.add_axes(rect)
    c = np.asarray(corr, dtype=float)
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    lo = np.floor(c[c < 1].min() * 100) / 100
    im = ax.imshow(c, cmap=cmap, vmin=lo, vmax=1.0)
    ax.set_xticks(range(len(names)), names, rotation=35, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.tick_params(length=0, labelsize=8.8, labelcolor=INK)
    for s in ax.spines.values():
        s.set_visible(False)
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j:
                ax.text(j, i, f"{c[i, j]:.3f}", ha="center", va="center", fontsize=8,
                        color="#ffffff" if c[i, j] > lo + (1 - lo) * 0.55 else INK)
    _title(ax, "Two views, two blocks", "Spearman correlation of the out-of-fold predictions")


def _gains(fig, rect, gains):
    """gains: list of (candidate, view, gain x 1e-4), any order."""
    ax = fig.add_axes(rect)
    g = sorted(gains, key=lambda t: t[2])
    yy = np.arange(len(g))
    ax.barh(yy, [v for _, _, v in g], height=0.55, color=[VIEW_COLORS[w] for _, w, _ in g], zorder=2)
    for yv, (_, _, v) in zip(yy, g):
        ax.text(v + (0.04 if v >= 0 else -0.04), yv, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right",
                fontsize=8.5, color=INK2)
    ax.set_yticks(yy, [n for n, _, _ in g])
    ax.tick_params(axis="y", length=0, labelsize=8.8, labelcolor=INK)
    ax.axvline(0, color=AXIS, lw=1)
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(min(0, min(v for *_, v in g)) - 0.35, max(v for *_, v in g) * 1.75)
    ax.set_xlabel("stack CV gain when added alone to Part 1 (× 1e-4)")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=c, label=k) for k, c in VIEW_COLORS.items()], loc="lower right",
              frameon=False, fontsize=8.5)
    _title(ax, "What each candidate adds", "one at a time on top of Part 1's five-model stack")


def _smoothing(fig, rect, sm):
    """sm: dict(weights=[...], cv=[...], kept=[weights in the blend])"""
    ax = fig.add_axes(rect)
    w, cv = np.asarray(sm["weights"]), np.asarray(sm["cv"])
    ax.plot(w, cv, color=BLUE, lw=2, zorder=2)
    for x, v in zip(w, cv):
        kept = x in sm["kept"]
        ax.scatter([x], [v], s=60 if kept else 46, color=BLUE if kept else SURFACE, edgecolor=BLUE if not kept else SURFACE,
                   linewidth=2, zorder=3)
        ax.annotate(f"{v:.6f}", (x, v), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8.5,
                    color=INK, fontweight="bold" if kept else "normal")
    ax.set_xscale("log")
    ax.set_xticks(w, [str(int(x)) for x in w])
    ax.minorticks_off()
    ax.set_xlim(w.min() / 1.6, w.max() * 1.6)
    span = cv.max() - cv.min()
    ax.set_ylim(cv.min() - span * 0.4, cv.max() + span * 0.6)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.5f}"))
    ax.set_xlabel("target-encoding smoothing weight (log scale)")
    ax.set_ylabel("5-fold CV AUC, 4-leaf LightGBM")
    ax.text(0.02, 0.04, "filled = kept in the blend", transform=ax.transAxes, fontsize=8.5, color=INK2)
    _title(ax, "Smoothing is a free extra view", "same model, different encodings: 100 and 300 both make the blend")


def _slope(fig, rect, parts, title="Part 1 → Part 2", sub="the public score moved with the CV"):
    """parts: list of (label, cv, lb)."""
    ax = fig.add_axes(rect)
    x = np.arange(len(parts))
    for k, (key, col, lab) in enumerate([(1, MUTED, "5-fold CV"), (2, BLUE, "public LB")]):
        v = [p[key] for p in parts]
        ax.plot(x, v, color=col, lw=2, zorder=2)
        ax.scatter(x, v, s=52, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        for xi, vi in zip(x, v):
            last = xi == x[-1]
            ax.annotate(f"{vi:.5f}" + (f"  {lab}" if last else ""), (xi, vi), xytext=(10 if last else -10, 0),
                        textcoords="offset points", ha="left" if last else "right", va="center", fontsize=9, color=INK)
    ax.set_xticks(x, [p[0] for p in parts])
    ax.tick_params(axis="x", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(-0.6, len(parts) - 1 + 1.5)
    allv = [p[1] for p in parts] + [p[2] for p in parts]
    span = max(allv) - min(allv)
    ax.set_ylim(min(allv) - span * 0.3, max(allv) + span * 0.3)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, title, sub)


def _weights(fig, rect, blend, sub="rank hill climbing, steps of 1/8"):
    """blend: list of (member, view, weight)."""
    ax = fig.add_axes(rect)
    ax.set_axis_off()
    bottom = 1.0
    for nm, view, w in blend:
        c = VIEW_COLORS[view]
        ax.add_patch(FancyBboxPatch((0.0, bottom - w + 0.006), 0.26, w - 0.012,
                                    boxstyle="round,pad=0,rounding_size=0.012", transform=ax.transAxes,
                                    facecolor=c, edgecolor="none"))
        ax.text(0.03, bottom - w / 2, f"{w:.1%}".replace(".0%", "%"), transform=ax.transAxes, va="center", fontsize=9.5,
                fontweight="bold", color=INK if c == YELLOW else "#ffffff")
        ax.text(0.31, bottom - w / 2, nm, transform=ax.transAxes, va="center", fontsize=9.2, color=INK)
        bottom -= w
    _title(ax, "The blend", sub)


def draw_part2(stats, names, corr, gains, smoothing, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _corr(fig, [0.075, 0.43, 0.25, 0.28], names, corr)
    _gains(fig, [0.48, 0.43, 0.17, 0.28], gains)
    _smoothing(fig, [0.73, 0.43, 0.22, 0.28], smoothing)
    _slope(fig, [0.075, 0.05, 0.36, 0.25], parts)
    _weights(fig, [0.60, 0.05, 0.30, 0.25], blend)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------- Part 3
DIV = ["#eb6834", "#f3a27d", "#f6c9b4", "#e9e8e3", "#b9d3f3", "#6da7ec", "#2a78d6"]   # less fraud ... more fraud


def _digits(fig, rect, dig):
    """dig: dict(cols=[...], rate=[[10 rates] per col], base=float)."""
    ax = fig.add_axes(rect)
    rate = np.asarray(dig["rate"], dtype=float)
    lift = rate / dig["base"] - 1
    lim = max(0.25, np.abs(lift).max())
    cmap = LinearSegmentedColormap.from_list("div", DIV)
    im = ax.imshow(lift, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    for i in range(lift.shape[0] + 1):
        ax.axhline(i - 0.5, color=PAGE, lw=3)
    for j in range(lift.shape[1] + 1):
        ax.axvline(j - 0.5, color=PAGE, lw=3)
    for i in range(lift.shape[0]):
        for j in range(lift.shape[1]):
            if abs(lift[i, j]) >= 0.10:
                ax.text(j, i, f"{rate[i, j]:.2%}", ha="center", va="center", fontsize=8.6,
                        color="#ffffff" if abs(lift[i, j]) > lim * 0.55 else INK)
    ax.set_xticks(range(10), [f".{d}" for d in range(10)])
    ax.set_yticks(range(len(dig["cols"])), dig["cols"])
    ax.tick_params(length=0, labelsize=10)
    ax.tick_params(axis="y", labelcolor=INK)
    ax.set_xlabel("first digit after the decimal point")
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, colors=MUTED, length=0)
    cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0%}"))
    cb.set_label("fraud rate vs base", color=MUTED, fontsize=8.5)
    _title(ax, "The digits leak the label",
           f"fraud rate by the first decimal digit · base {dig['base']:.2%} · the same digits stand out in every column")


def _steps(fig, rect, steps, title, sub, xlabel):
    """steps: list of (label, auc); bars show the gain over the first step (× 1e-4)."""
    ax = fig.add_axes(rect)
    lab = [s for s, _ in steps]
    v = np.array([a for _, a in steps])
    yy = np.arange(len(lab))[::-1]
    gain = (v - v[0]) * 1e4
    ax.barh(yy, np.maximum(gain, 0.3), height=0.5, color=[AXIS] + [BLUE] * (len(v) - 1), zorder=2)
    for yv, g, a in zip(yy, gain, v):
        ax.text(max(g, 0) + gain.max() * 0.02, yv, f"{a:.5f}" + (f"   +{g:.1f}" if g > 0 else "   start"),
                va="center", fontsize=9.5, color=INK)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(0, gain.max() * 1.5)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(xlabel)
    _title(ax, title, sub)


def _families(fig, rect, fam, left="Part 2 features", right="+ digit keys", title="Every model family gains",
              sub="5-fold CV AUC, same folds · (gain × 1e-4)", dec=0):
    """fam: list of (model family, CV before, CV after), best first."""
    ax = fig.add_axes(rect)
    names = [f for f, _, _ in fam][::-1]
    a = np.array([x for _, x, _ in fam][::-1])
    b = np.array([x for _, _, x in fam][::-1])
    yy = np.arange(len(names))
    ax.hlines(yy, a, b, color=AXIS, lw=2, zorder=1)
    ax.scatter(a, yy, s=60, color=MUTED, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.scatter(b, yy, s=70, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
    for yv, x0, x1 in zip(yy, a, b):
        ax.text(x1 + 0.00012, yv, f"{x1:.5f}  (+{(x1 - x0) * 1e4:.{dec}f})", va="center", fontsize=9, color=INK)
    ax.set_yticks(yy, names)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(min(a.min(), b.min()) - 0.0004, max(b) + 0.0019)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.{3 if dec == 0 else 4}f}"))
    if left:
        ax.text(a[-1], yy[-1] + 0.45, left, ha="center", fontsize=8.5, color=MUTED)
        ax.text(b[-1], yy[-1] + 0.45, right, ha="center", fontsize=8.5, color=INK2)
    ax.set_ylim(-0.6, len(names) - 0.2)
    _title(ax, title, sub)


def draw_part3(stats, digits, steps, families, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _digits(fig, [0.155, 0.45, 0.40, 0.24], digits)
    _steps(fig, [0.72, 0.45, 0.17, 0.24], steps, "Four keys per number",
           "one 4-leaf LightGBM, fold 0", "AUC gain over per-value TE alone (× 1e-4)")
    _families(fig, [0.155, 0.06, 0.30, 0.25], families)
    _slope(fig, [0.53, 0.06, 0.25, 0.25], parts, "Part 1 → Part 3", "public LB follows the CV")
    _weights(fig, [0.84, 0.06, 0.13, 0.25], blend,
             f"rank hill climbing, steps of 1/{round(1 / min(w for _, _, w in blend))}")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------- Part 4
def _csweep(fig, rect, sw):
    """sw: dict(C=[...], series={label: [fold-0 AUC or None per C]}, ref=(label, auc))."""
    ax = fig.add_axes(rect)
    C = np.asarray(sw["C"], dtype=float)
    for (lab, au), col in zip(sw["series"].items(), [MUTED, BLUE]):
        au = np.array([np.nan if v is None else v for v in au], dtype=float)
        ok = ~np.isnan(au)
        ax.plot(C[ok], au[ok], color=col, lw=2, zorder=2)
        ax.scatter(C[ok], au[ok], s=46, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        k = int(np.nanargmax(au))
        ax.annotate(f"{au[k]:.5f}", (C[k], au[k]), xytext=(0, 9 if col == BLUE else -15), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold", color=INK)
        last = np.where(ok)[0][-1]
        ax.text(C[last] * 1.25, au[last], lab, va="center", fontsize=9, color=INK2)
    rl, rv = sw["ref"]
    ax.axhline(rv, color=AXIS, lw=1.2, ls=(0, (4, 3)), zorder=1)
    ax.text(C.max() * 1.25, rv, rl, va="center", fontsize=8.5, color=MUTED)
    ax.set_xscale("log")
    tk = [c for c in C if c != 0.005]
    ax.set_xticks(tk, [f"{c:g}" for c in tk])
    ax.minorticks_off()
    ax.set_xlim(C.min() / 1.6, C.max() * 9)
    vals = [v for a in sw["series"].values() for v in a if v is not None] + [rv]
    span = max(vals) - min(vals)
    ax.set_ylim(min(vals) - span * 0.08, max(vals) + span * 0.18)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.set_xlabel("inverse L2 strength C (log scale)")
    ax.set_ylabel("fold-0 AUC")
    _title(ax, "One coefficient per value, fitted jointly",
           "logistic regression with every value and digit key one-hot, plus the TE logits")


def draw_part4(stats, steps, csweep, families, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _steps(fig, [0.20, 0.45, 0.25, 0.24], steps, "Same models, more data",
           "public LB · step 2 changes only the test predictions", "public LB gain over Part 3 (× 1e-4)")
    _csweep(fig, [0.56, 0.45, 0.28, 0.24], csweep)
    _families(fig, [0.20, 0.06, 0.25, 0.25], families, left=None, right=None, dec=1,
              title="Steadier members", sub="grey = Part 3 member, blue = Part 4 · 5-fold CV (gain × 1e-4)")
    _slope(fig, [0.53, 0.06, 0.25, 0.25], parts, "Part 1 → Part 4", "public LB and 5-fold CV")
    _weights(fig, [0.84, 0.06, 0.13, 0.25], blend,
             f"rank hill climbing, steps of 1/{round(1 / min(w for _, _, w in blend))}")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def _leaves(fig, rect, lv):
    """lv: dict(seeds={leaves: [5-fold CV per seed]}, bag={leaves: CV of the seed mean})."""
    ax = fig.add_axes(rect)
    ks = sorted(lv["seeds"], key=int)
    for i, k in enumerate(ks):
        s = lv["seeds"][k]
        ax.scatter(np.full(len(s), i) + np.linspace(-0.08, 0.08, len(s)), s, s=40, color=MUTED, edgecolor=SURFACE,
                   linewidth=1.5, zorder=3)
        if k in lv["bag"]:
            b = lv["bag"][k]
            ax.hlines(b, i - 0.22, i + 0.22, color=BLUE, lw=2.5, zorder=4)
            ax.text(i + 0.28, b, f"{b:.5f}", va="center", fontsize=9.5, fontweight="bold", color=INK)
    allv = [v for s in lv["seeds"].values() for v in s]
    ax.text(0.14, lv["seeds"][ks[0]][0], f"{lv['seeds'][ks[0]][0]:.5f}", va="center", fontsize=9, color=INK2)
    ax.set_xticks(range(len(ks)), [f"{k} leaves" for k in ks])
    ax.tick_params(axis="x", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(-0.5, len(ks) - 0.1)
    span = max(allv) - min(allv)
    ax.set_ylim(min(allv) - span * 0.2, max(allv) + span * 0.35)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, "Three leaves, not four", "5-fold CV · grey = one seed, blue = mean of 3 seeds")


def _deltas(fig, rect, rows, title, sub):
    """rows: list of (label, auc); bars show the change from the first row (× 1e-4)."""
    ax = fig.add_axes(rect)
    ref = rows[0][1]
    lab = [r for r, _ in rows[1:]]
    d = np.array([(a - ref) * 1e4 for _, a in rows[1:]])
    yy = np.arange(len(lab))[::-1]
    ax.barh(yy, d, height=0.55, color=[BLUE if v > 0 else ORANGE for v in d], zorder=2)
    for yv, v in zip(yy, d):
        ax.text(v + (0.4 if v >= 0 else -0.4), yv, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right",
                fontsize=9.5, color=INK)
    ax.axvline(0, color=INK2, lw=1, zorder=3)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_xlim(d.min() * 1.35, max(d.max(), 0) + 6)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(f"fold-0 AUC change vs the plain GLM ({ref:.5f}), × 1e-4")
    _title(ax, title, sub)


def _cvlb(fig, rect, pts):
    """pts: list of (label, cv, lb, group); group 'glm' = contains the plain GLM."""
    ax = fig.add_axes(rect)
    for lab, cv, lb, g in pts:
        col = ORANGE if g == "glm" else BLUE
        big = "Part 5" in lab
        ax.scatter(cv, lb, s=150 if big else 70, color=col, edgecolor=INK if big else SURFACE, linewidth=1.5 if big else 2,
                   zorder=4 if big else 3)
    off = {"Part 3 refit": (8, -4), "Part 4": (-8, 6), "5 seeds": (7, 7), "+ EBM i25": (8, -3),
           "3 leaves": (4, -10), "plain GLM swap": (8, 0), "CV best, refit": (-8, 8),
           "CV best, refit + folds": (-8, -12), "half plain GLM": (-8, -7), "CV best without it (Part 5)": (10, 0)}
    for lab, cv, lb, g in pts:
        dx, dy = off.get(lab, (8, 0))
        ax.annotate(lab, (cv, lb), xytext=(dx, dy), textcoords="offset points", fontsize=9, color=INK,
                    ha="left" if dx > 0 else "right", va="center")
    cvs, lbs = [p[1] for p in pts], [p[2] for p in pts]
    ax.set_xlim(min(cvs) - 0.00006, max(cvs) + 0.00005)
    ax.set_ylim(min(lbs) - 0.00007, max(lbs) + 0.00007)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.5f}"))
    ax.set_xlabel("5-fold CV AUC (700k rows)")
    ax.set_ylabel("public LB (60k rows)")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, color=c, label=t) for c, t in
                       [(BLUE, "without the plain GLM"), (ORANGE, "with the plain GLM")]],
              loc="lower left", frameon=False, fontsize=9, handletextpad=0.2)
    _title(ax, "The CV and the public board disagree", "every blend we submitted after Part 3")


def _refit(fig, rect, rows):
    """rows: list of (held-out fold, fold average, refit, rank mean); bars = change vs the fold average."""
    ax = fig.add_axes(rect)
    x = np.arange(len(rows))
    b = np.array([(r[2] - r[1]) * 1e4 for r in rows])
    c = np.array([(r[3] - r[1]) * 1e4 for r in rows])
    ax.bar(x - 0.19, b, width=0.34, color=MUTED, zorder=2)
    ax.bar(x + 0.19, c, width=0.34, color=BLUE, zorder=2)
    for xi, v in zip(np.r_[x - 0.19, x + 0.19], np.r_[b, c]):
        ax.text(xi, v + (0.15 if v >= 0 else -0.15), f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top",
                fontsize=8.8, color=INK)
    ax.axhline(0, color=INK2, lw=1, zorder=3)
    ax.set_xticks(x, [f"held out\n{r[0]}" for r in rows])
    ax.tick_params(axis="x", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_ylim(min(b.min(), c.min()) - 0.8, max(b.max(), c.max()) + 0.8)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("AUC change vs fold average (× 1e-4)")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=MUTED, label="refit on all rows"), Patch(color=BLUE, label="rank mean of both")],
              loc="lower left", frameon=False, fontsize=9)
    _title(ax, "Refit or fold average?", "3-leaf trees, 3 seeds, scored on a held-out fold")


def draw_part5(stats, leaves, glm_design, cvlb, refit, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _leaves(fig, [0.06, 0.45, 0.28, 0.24], leaves)
    _deltas(fig, [0.60, 0.45, 0.30, 0.24], glm_design, "A plain GLM is the strongest GLM",
            "logistic regression on the TE logits · what each extra block does")
    _cvlb(fig, [0.06, 0.06, 0.40, 0.27], cvlb)
    _refit(fig, [0.53, 0.06, 0.22, 0.25], refit)
    _weights(fig, [0.83, 0.06, 0.14, 0.25], blend, "best CV without the plain GLM, 1/9 steps")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


draw_part5(
    stats=[('Public LB', '0.95160', '+0.00006 over Part 4'), ('5-fold CV AUC', '0.95092', '+0.00013 over Part 4'), ('Plain GLM, CV', '0.95051', "+0.0006 over Part 4's GLM"), ('Its public verdict', '−0.00018', 'swapped into Part 4: CV +0.00013')],
    leaves={'seeds': {'2': [0.950204575897117], '3': [0.9506444964240918, 0.9503154204418328, 0.9503331027872026], '4': [0.9503999740916812, 0.9501976208180694, 0.9502346223233641]}, 'bag': {'3': 0.950626761687232, '4': 0.9505103584240805}},
    glm_design=[('TE logits + raw columns (plain)', 0.949617), ('+ TE smoothing 50 instead of 20', 0.949759), ('+ splines', 0.949316), ('+ one-hot of values (≤ 1000 distinct)', 0.947958), ('+ both (the Part 1-3 GLM design)', 0.947762), ('+ GPT-2 token keys', 0.949509), ('+ one-hot values and keys, C 0.01 (Part 4)', 0.949368)],
    cvlb=[('Part 3 refit', 0.950728, 0.95132, 'base'), ('Part 4', 0.95079, 0.95154, 'base'), ('5 seeds', 0.950846, 0.95151, 'base'), ('+ EBM i25', 0.950883, 0.95145, 'base'), ('3 leaves', 0.950818, 0.95154, 'base'), ('plain GLM swap', 0.950914, 0.95136, 'glm'), ('CV best, refit', 0.951032, 0.95132, 'glm'), ('CV best, refit + folds', 0.951032, 0.95119, 'glm'), ('half plain GLM', 0.950865, 0.95147, 'glm'), ('CV best without it (Part 5)', 0.950924, 0.9516, 'base')],
    refit=[('fold 0', 0.949717, 0.949734, 0.949807), ('fold 1', 0.951252, 0.951252, 0.951336), ('fold 2', 0.955742, 0.955561, 0.95572)],
    blend=[('lgb_diglead_l3_bag', 'target-encoded view', 0.2222), ('glm_ohk2_bag', 'joint GLM', 0.2222), ('ebm_i25_bag4', 'target-encoded view', 0.2222), ('lgb_dig_s100_bag5', 'target-encoded view', 0.1111), ('lgb_rawdig_l4', 'raw view', 0.1111), ('lgb_dig_l7', 'target-encoded view', 0.1111)],
    title="S6E11 Credit Card Fraud  ·  Part 5",
    subtitle="Three leaves, a plain GLM, and a public board that disagrees with the CV")
plt.show()

# S6E11 Part 5 | 3 Leaves, Plain GLM | LB 0.95160

> **This notebook grows.** Every part is a new version of this same notebook: one new idea on top of the
> previous ones, measured on the same folds, with the whole history kept below — what was tried, what each
> idea was worth, and what did not work. If you upvoted an earlier part, thank you; the newest part is
> always at the top.

| part | idea | 5-fold CV | public LB |
|---|---|---|---|
| **Part 1** | every value of every column has its own fraud rate (out-of-fold TE); 4-leaf LightGBM, GLMs, RealMLP, logistic stack | 0.94892 | 0.94928 |
| **Part 2** | raw-view trees next to target-encoded ones, TE smoothing as an extra view, rank hill climbing | 0.94923 | 0.94945 |
| **Part 3** | the generator writes numbers digit by digit: target-encoded cents, int(x), int(x/10), int(10x) and leading digits; every family retrained; EBMs | 0.95073 | 0.95113 |
| **Part 4** | refit every member on all training rows, three-seed bags, a GLM fitting one-hot values and digit keys jointly with the encodings | 0.95079 | 0.95154 |
| **Part 5** (this version) | 3-leaf trees; a plain GLM on the encodings (CV 0.95051) that the public board rejects; the best-CV blend without it; refit vs fold average on held-out folds | 0.95092 | 0.95160 |

## What is new in Part 5

**Public LB 0.95160 (our best, +0.00006 over Part 4) · 5-fold CV 0.95092** — two findings that hold up
on 700k out-of-fold rows, and a public board that accepts one of them and rejects the other.

1. **Three leaves, not four.** On five folds the 3-leaf LightGBM beats the 4-leaf one: 0.95028 → 0.95043 on
   average over three seeds, 0.95051 → **0.95063** as a three-seed bag. Fold 0 alone said the opposite
   (−0.00003), which is why Part 1 kept four leaves. Swapped into Part 4's blend with the same weights: CV
   +0.00003, public LB unchanged (0.95154).
2. **A plain GLM is the strongest GLM.** A logistic regression on the raw columns, the flags, the one-hot
   categories and the target-encoding logits of every value and digit key, *without* the splines and value
   one-hots our GLMs have carried since Part 1, scores **0.95051** (three seeds, TE
   smoothing 50): above Part 4's joint GLM (0.94991) and level with the 4-leaf LightGBM bag.
   The extra blocks were costing 0.0019 on fold 0. The blend with the best CV gives it the largest
   weight: CV **0.95103**.
3. **The public board says no.** Every blend with the plain GLM scored lower on the public board: the
   CV-best blend 0.95132 (CV +0.00024 over Part 4), and Part 4 with only its GLM swapped 0.95136 (CV
   +0.00013, public noise of that swap about ±0.0001), half of its weight 0.95147. Without the plain GLM
   the best-CV blend scores **0.95160**, our best public score: the notebook's `submission.csv`. We cannot
   yet tell whether the CV or the public board is right about the plain GLM; the private board (80% of the
   test rows) will. The CV-best blend with it is written to `submission_cv_best.csv` (5.5, 5.6).
4. **Refit or fold average, checked properly.** Part 4 switched to test predictions from one refit on all
   rows after a single public-board comparison. On held-out folds of the training data the refit alone is
   no better than the fold average (−0.00005), and the rank mean of the two is best (+0.00005) (5.4).
5. GPT-2 token keys do not help (5.3), and LightGBM column sampling makes no difference to the digit-key
   gain (a question from the comments).

| member | what it is | 5-fold CV AUC | blend weight |
|---|---|---|---|
| `lgb_diglead_l3_bag` | 3-leaf LightGBM + leading digits, mean of 3 seeds (new) | 0.95063 | 2/9 |
| `glm_ohk2_bag` | joint GLM, more key levels, TE smoothing 50, 2 TE seeds (after Part 4) | 0.95012 | 2/9 |
| `ebm_i25_bag4` | EBM + leading digits, 25 interactions, 4 seeds (after Part 4) | 0.95026 | 2/9 |
| `lgb_dig_s100_bag5` | 4-leaf LightGBM, TE smoothing 100, mean of 5 seeds (after Part 4) | 0.95049 | 1/9 |
| `lgb_rawdig_l4` | 4-leaf LightGBM, raw columns + decimals (Part 3) | 0.94837 | 1/9 |
| `lgb_dig_l7` | 7-leaf LightGBM with the digit keys (Part 3) | 0.95016 | 1/9 |
| **blend** (test: refit predictions, as in Part 4) | | **0.95092** | |

**Part 4 in one paragraph:** the same signal with less noise: test predictions from one more model per member
trained on every training row (the Part 3 blend went from LB 0.95113 to 0.95132), three-seed bags, and a GLM
that fits one-hot values and digit keys jointly with the encodings. CV 0.95079, LB 0.95154.

**Part 3 in one paragraph:** the synthetic generator writes numbers digit by digit and the digits carry the
label; four target-encoded keys per float column (cents, `int(x)`, `int(x/10)`, `int(10x)`) lift every model
family by 0.0008-0.0017. CV 0.95073, LB 0.95113.

**Part 2 in one paragraph:** shallow trees on the *raw* columns next to the target-encoded ones; the TE
smoothing weight as a free extra view; rank hill climbing. Six members: CV 0.94923, LB 0.94945.

**Part 1 in one paragraph:** the original 20k rows are almost a logistic model; out-of-fold target
encoding of *every* column (each distinct number is a category) lifts every model family; with those
encodings the best trees have 4 leaves. Five models stacked: CV 0.94892, LB 0.94928.

## Credits and thanks

| Whose work | What we took from it |
|---|---|
| **Udit Jain** — [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) | The original dataset this competition was generated from (Part 1, section 2; value counts in Part 2). |
| **aphoristicartist** — [S6E11 Fraud: EDA + 3 GBDTs + Blend](https://www.kaggle.com/code/aphoristicartist/s6e11-fraud-eda-3-gbdts-blend-lb-0-9468) | The day-one baseline we measured against (3 GBDTs, OOF ≈ 0.9463, LB 0.9468), the same fold scheme, and the reminder that the public board scores only ~20% of the test rows. |
| **Will** — [S6E11 Refined LightGBM](https://www.kaggle.com/code/wguesdon/s6e11-refined-lightgbm) | A 15-leaf, heavily regularised LightGBM — the first hint that small trees win here. We pushed it down to 4 leaves. |
| **Vladimir Demidov** — [PS S6E10 RealMLP · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) | The RealMLP hyper-parameters (from last month's competition), used here almost unchanged. |
| **David Holzmüller et al.** — [pytabkit](https://github.com/dholzmueller/pytabkit) | RealMLP itself. |
| **Ravi Ramakrishnan** — [PlaygroundS6E11 Public EDA V1](https://www.kaggle.com/code/ravi20076/playgrounds6e11-public-eda-v1) | Target encodings of rounded values: the idea behind Part 3's `int(x)` and `int(x/10)` keys. |
| **alex chilton** — [Credit Card Fraud: Metric, EDA, 4-Model Ensemble](https://www.kaggle.com/code/alexchilton/credit-card-fraud-metric-eda-4-model-ensemble) | The first decimal digit and rounded values as features (Part 3). We measured the two-decimal `cents` key next to them. |
| **InterpretML** — [Explainable Boosting Machine](https://github.com/interpretml/interpret) | The EBM in Part 3. |
| **Bulşah Keçici** — [S6E11 \| Value TE + Digit & Token Keys \| LR Stack](https://www.kaggle.com/code/bulsahkecici/s6e11-value-te-digit-token-keys-lr-stack) | A logistic regression on the encoding logits without extra blocks (their CV 0.95035): the plain GLM of Part 5. The GPT-2 token keys we tested in 5.3. |
| **Akmal Xodarev** — [S6E11 \| Token Keys, EBM, Full Refit \| CV 0.95079](https://www.kaggle.com/code/busyaprime/s6e11-token-keys-ebm-full-refit-cv-0-95079) | 3 leaves beating 4 on five folds (Part 5). |
| **Masaya Kawamata** — [S6E11 \| Five XGBoost folds beat one full fit](https://www.kaggle.com/code/masayakawamata/s6e11-five-xgboost-folds-beat-one-full-fit); **Georgy Mamarin** — [S6E11 Fraud \| LB 0.95095 \| Shallow Digit Models](https://www.kaggle.com/code/georgymamarin/s6e11-fraud-lb-0-95095-shallow-digit-models) | Held-out comparisons of fold averages and full refits, which made us re-check Part 4's refit (5.4). Georgy also asked about column sampling in the comments. |

All code below is our own; ideas from the notebooks above are marked where they are used. Thank you!

## Setup and data

Inputs: the competition data, the original dataset `uditjain13/credit-card-fraud-detection-2026`, and our
dataset `goodpjw2008/s6e11-fraud-series-artifacts` (RealMLP predictions trained on a GPU, EBM predictions
trained on 16 CPU cores, and the bagged and refit members of Parts 4 and 5). **CPU only**, about 4 hours for
every part together. Set `RETRAIN_NN = True` (and turn the GPU on) to retrain the RealMLPs, `RETRAIN_EBM = True`
(Part 3) to retrain the EBMs, `RETRAIN_P4 = True` (Part 4) and `RETRAIN_P5 = True` (Part 5) to retrain the
seed bags, the GLMs and every refit here (several more hours).

24 features: 13 numeric, 6 yes/no flags, 5 categoricals; no missing values. About 1 row in 60 is fraud.
Every model in every part uses the same `StratifiedKFold(5, shuffle=True, random_state=42)` split.

In [ ]:
import subprocess, sys
from importlib.metadata import version

# The LB 0.94945 submission was trained with XGBoost 3.2.0. The newer release on Kaggle grows slightly
# different trees from the same data (xgb_te_d2 CV 0.94872 instead of 0.94881), so pin it to reproduce.
if version("xgboost") != "3.2.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "xgboost==3.2.0"], check=True)

import time
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from scipy import sparse
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

INPUT = os.environ.get("INPUT_DIR", "/kaggle/input")


def find(name, hint=""):
    hits = sorted(glob.glob(os.path.join(INPUT, "**", name), recursive=True))
    return ([h for h in hits if hint in h] or hits)[0]


SMOKE = os.environ.get("SMOKE") == "1"     # tiny dry run of the whole notebook
RETRAIN_NN = False                         # True: retrain RealMLP (needs pytabkit + a GPU)
SEED, N_FOLDS, TARGET = 42, 5, "is_fraud"

CATS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLAGS = ["is_foreign_transaction", "is_new_merchant", "used_vpn",
         "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMS = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
        "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
        "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes"]
FEATS = NUMS + FLAGS + CATS

tr = pd.read_csv(find("train.csv", "playground-series-s6e11"))
te = pd.read_csv(find("test.csv", "playground-series-s6e11"))
og = pd.read_csv(find("credit_card_fraud_2026.csv")).drop(columns="transaction_id")
if SMOKE:
    tr, te = tr.iloc[:40000].copy(), te.iloc[:10000].copy()
for df in (tr, te, og):
    df[FLAGS] = df[FLAGS].astype("int8")
y = tr[TARGET].astype(int).values
og[TARGET] = og[TARGET].astype(int)
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))

full = pd.concat([tr[FEATS], te[FEATS]], ignore_index=True)
allv = pd.concat([tr[FEATS], te[FEATS], og[FEATS]], ignore_index=True)
for c in CATS:
    full[c] = pd.Categorical(full[c], categories=sorted(allv[c].unique()))
Xtr, Xte = full.iloc[:len(tr)].reset_index(drop=True), full.iloc[len(tr):].reset_index(drop=True)
OOF, PRED, FULL, ITS = {}, {}, {}, {}      # out-of-fold, 5-fold-average test, refit test (Part 4), best iterations


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


print(f"train {tr.shape}, test {te.shape}, original {og.shape}")
print(f"fraud rate: train {y.mean():.4%}, original {og[TARGET].mean():.4%}")

---
# Part 1 — Per-value target encoding and 4-leaf trees

*Public LB 0.94928 · 5-fold CV 0.94892 · first version of this notebook*

In [ ]:
_y = tr[TARGET]
_rx = pd.qcut(tr["merchant_risk_score"].rank(method="first"), 10, labels=False)
_vy = pd.qcut(tr["velocity_score"].rank(method="first"), 10, labels=False)
draw(
    stats=[("Public LB", "0.94928", "#1 on the board when submitted (Nov 1)"),
           ("5-fold CV AUC", "0.94892", "+0.0019 over a 31-leaf LightGBM"),
           ("Fraud rate", f"{_y.mean():.2%}", f"{int(_y.sum()):,} frauds in {len(_y):,} rows"),
           ("Stack", "5 models", "LightGBM · 2 GLMs · 2 RealMLPs")],
    heat=dict(rate=_y.groupby([_vy, _rx]).mean().unstack().values,
              xname="merchant_risk_score", yname="velocity_score", base=_y.mean()),
    te_gain=[['account_balance_usd', 0.5073, 0.5375], ['hours_since_last_txn', 0.5582, 0.5808], ['customer_age', 0.6278, 0.6487], ['amount_usd', 0.5412, 0.5553], ['distance_from_home_km', 0.5023, 0.5113]],
    depth=dict(leaves=[2, 4, 7, 15, 31, 63], series={'raw features': [0.945423, 0.945879, 0.945708, 0.944967, 0.944632, 0.943917], '+ target encoding': [0.947167, 0.947402, 0.946927, 0.946777, 0.946502, 0.945609]}),
    ladder=[('LightGBM, 31 leaves', 0.947061), ('4 leaves', 0.947772), ('+ per-value target encoding', 0.948845), ('stack with 2 GLMs + 2 RealMLPs', 0.94892)],
    blend=[('LightGBM, 4 leaves + TE', 0.6), ('RealMLP ×2', 0.31), ('GLM ×2', 0.09)],
    title="S6E11 Credit Card Fraud  ·  Part 1",
    subtitle="Per-value target encoding + 4-leaf LightGBM + GLM + RealMLP, stacked")
plt.show()

## 1.1 The original data is a logistic model

Fit two models on the 20k original rows only, scored by their own 5-fold CV. If the original labels were
drawn from a logistic formula, the linear model should win, and it does by a wide margin
(**0.9435** vs 0.9302). The coefficients read like a rulebook: velocity, CVV
retries, merchant risk and the mismatch flags push fraud up; biometric or PIN authentication pushes it down.

In [ ]:
def onehot_frame(df):
    X = pd.get_dummies(df[FEATS], columns=CATS, dtype=float)
    return (X - X.mean()) / X.std()


Xo = onehot_frame(og)
p_lr, p_gb = np.zeros(len(og)), np.zeros(len(og))
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(Xo, og[TARGET]):
    p_lr[b] = LogisticRegression(C=1.0, max_iter=3000).fit(Xo.iloc[a], og[TARGET].iloc[a]).predict_proba(Xo.iloc[b])[:, 1]
    gb = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                            subsample=0.85, subsample_freq=1, colsample_bytree=0.85, reg_lambda=5, verbosity=-1)
    p_gb[b] = gb.fit(Xo.iloc[a], og[TARGET].iloc[a]).predict_proba(Xo.iloc[b])[:, 1]
print(f"original data, 5-fold AUC: logistic regression {roc_auc_score(og[TARGET], p_lr):.4f} | "
      f"LightGBM {roc_auc_score(og[TARGET], p_gb):.4f}")

coef = pd.Series(LogisticRegression(C=1.0, max_iter=3000).fit(Xo, og[TARGET]).coef_[0], index=Xo.columns)
top = coef.reindex(coef.abs().sort_values().index[-14:])
_style()
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(range(len(top)), top.values, height=0.55, color=[BLUE if v > 0 else ORANGE for v in top.values], zorder=2)
ax.set_yticks(range(len(top)), top.index)
ax.tick_params(axis="y", length=0, labelcolor=INK)
ax.axvline(0, color=AXIS, lw=1)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True); ax.spines["left"].set_visible(False)
ax.set_xlabel("logistic-regression coefficient (standardized inputs)")
_title(ax, "The original data reads like a scorecard", "blue raises the fraud odds, orange lowers them · top 14 of 54 terms")
plt.show()

## 1.2 Every value has its own risk

Treat every column as a category — each distinct value of `account_balance_usd` (43k values),
`hours_since_last_txn` (3.6k), `customer_age` (64), ... — and encode it by its fraud rate, **out of fold**:
inside each training fold, a 5-fold inner split gives the training rows their encodings, and the validation
fold and test get encodings fitted on the whole training fold. Smoothing toward the prior with weight 20
keeps rare values near the base rate.

The numbers that look useless on their own (`account_balance_usd` 0.507) carry signal value by value: the
synthetic generator gave individual values their own fraud rate, which no smooth function of the number
can recover. Value counts carry a weaker form of the same signal (+0.0008 for a 31-leaf LightGBM on fold 0).

In [ ]:
def add_te(Xa, ya, others, cols, smooth=20.0, inner=5, seed=0):
    """Out-of-fold target encoding of `cols` (each value = a category), appended as te_<col>."""
    prior = ya.mean()
    Xa, others = Xa.copy(), [o.copy() for o in others]
    key = lambda X, c: X[c].astype(str).values if str(X[c].dtype) == "category" else X[c].values
    rate = lambda k, t: (lambda s: (s["sum"] + prior * smooth) / (s["count"] + smooth))(
        pd.DataFrame({"k": k, "y": t}).groupby("k")["y"].agg(["sum", "count"]))
    for c in cols:
        ka, enc = key(Xa, c), np.zeros(len(Xa), dtype=np.float32)
        for i, j in StratifiedKFold(inner, shuffle=True, random_state=seed).split(ka, ya):
            enc[j] = pd.Series(ka[j]).map(rate(ka[i], ya[i])).fillna(prior).values
        Xa[f"te_{c}"] = enc
        r = rate(ka, ya)
        for o in others:
            o[f"te_{c}"] = pd.Series(key(o, c)).map(r).fillna(prior).values.astype(np.float32)
    return Xa, others


a, b = FOLDS[0]
_, (enc_b,) = add_te(Xtr.iloc[a], y[a], [Xtr.iloc[b]], FEATS)
rows = []
for c in FEATS:
    raw = roc_auc_score(y[b], enc_b[f"te_{c}"] if c in CATS else enc_b[c])
    rows.append((c, max(raw, 1 - raw), roc_auc_score(y[b], enc_b[f"te_{c}"])))
gain = pd.DataFrame(rows, columns=["column", "raw AUC", "encoded AUC"]).assign(gain=lambda d: d["encoded AUC"] - d["raw AUC"])
gain = gain.sort_values("gain", ascending=False)
display(gain.head(8).round(4).reset_index(drop=True))
_style()
fig = plt.figure(figsize=(9, 5))
_dumbbell(fig, [0.25, 0.1, 0.68, 0.72], [tuple(r) for r in gain.head(8)[["column", "raw AUC", "encoded AUC"]].values])
plt.show()

## 1.3 LightGBM with 4 leaves

Once the encodings are in, depth hurts — and even without them. Fold-0 AUC by tree size, every other
setting equal (learning rate 0.05, `min_child_samples=100`):

| num_leaves | raw features | + target encoding |
|---|---|---|
| 2 | 0.94542 | 0.94717 |
| 4 | 0.94588 | 0.94740 |
| 7 | 0.94571 | 0.94693 |
| 15 | 0.94497 | 0.94678 |
| 31 | 0.94463 | 0.94650 |
| 63 | 0.94392 | 0.94561 |

The final model uses `num_leaves=4`, learning rate 0.03 and early stopping on the validation fold.

In [ ]:
LGB = dict(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, max_bin=255, n_jobs=-1, verbose=-1)


def run_lgb(name, XA, XT, use_te, smooth=20.0):
    oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if use_te:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], FEATS, smooth)
        m = lgb.LGBMClassifier(**LGB, random_state=SEED)
        m.fit(Xa, y[a], eval_set=[(Xb, y[b])], eval_metric="auc", callbacks=[lgb.early_stopping(300, verbose=False)])
        oof[b] = m.predict_proba(Xb)[:, 1]
        pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {m.best_iteration_} trees ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name] = oof, pred
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")


run_lgb("lgb_te_l4", Xtr, Xte, use_te=True)

## 1.4 Two GLMs

A logistic regression on a sparse design: standardized numbers (log for money and hours), cubic B-splines
with quantile knots, the flags, one-hot categoricals, and every numeric column with at most 1000 distinct
values one-hot encoded (values seen fewer than 50 times are pooled). `glm_te` adds the standardized logits
of the target encodings and is regularized harder (`C=0.3`); `glm_sb` has no encodings.

In [ ]:
def design(max_card, min_freq=50, knots=8):
    num = full[NUMS].astype(float).copy()
    for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"):
        num[c] = np.log1p(num[c])
    z = StandardScaler().fit_transform(num)
    small = [c for c in NUMS if full[c].nunique() <= max_card]
    return sparse.hstack([
        sparse.csr_matrix(z), sparse.csr_matrix(full[FLAGS].values.astype(float)),
        OneHotEncoder(handle_unknown="ignore").fit_transform(full[CATS].astype(str)),
        sparse.csr_matrix(SplineTransformer(n_knots=knots, degree=3, knots="quantile").fit_transform(z)),
        OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq).fit_transform(full[small]),
    ]).tocsr()


def run_glm(name, max_card, C, with_te, src=None, te_cols=FEATS, extra=None, refit=False, base=None, smooth=20.0):
    X = design(max_card) if base is None else base                       # Part 5: a plain design without splines
    X = X if extra is None else sparse.hstack([X, extra]).tocsr()        # Part 4: + one-hot keys
    XA, XT = X[:len(tr)], X[len(tr):]
    SA, ST = src if src is not None else (Xtr, Xte)      # frames whose te_cols get encoded (Part 3 adds digit keys)

    def fit(a, others):
        Ma, Ms = XA[a], [M for M, _ in others]
        if with_te:
            ta, enc = add_te(SA.iloc[a], y[a], [S for _, S in others], te_cols, smooth)
            cols = [f"te_{c}" for c in te_cols]
            sc = StandardScaler().fit(logit(ta[cols].values))
            Ma, *Ms = (sparse.hstack([M, sparse.csr_matrix(sc.transform(logit(T[cols].values)))]).tocsr()
                       for M, T in zip([Ma] + Ms, [ta] + enc))
        m = LogisticRegression(C=C, max_iter=3000, tol=1e-6).fit(Ma, y[a])
        return [m.predict_proba(M)[:, 1] for M in Ms]

    oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
    for f, (a, b) in enumerate(FOLDS):
        oof[b], p = fit(a, [(XA[b], SA.iloc[b]), (XT, ST)])
        pred += p / N_FOLDS
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name] = oof, pred
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")
    if refit:                                            # Part 4: one more fit on every training row
        FULL[name], = fit(np.arange(len(tr)), [(XT, ST)])


run_glm("glm_te", max_card=1000, C=0.3, with_te=True)
run_glm("glm_sb", max_card=100, C=1.0, with_te=False)

## 1.5 RealMLP

RealMLP from pytabkit with the hyper-parameters of Vladimir Demidov's S6E10 notebook (4 ensemble members,
4 epochs, no early stopping). `realmlp_te` gets the target encodings as extra numeric inputs (+0.0019 CV over
plain `realmlp_v1`). Both were trained on our GPU with the same folds; their out-of-fold and test
predictions are loaded from our dataset. The training code is below (`RETRAIN_NN = True`).

In [ ]:
REALMLP = dict(
    n_ens=4, n_epochs=4, batch_size=512, use_early_stopping=False,
    lr=0.04, wd=0.0236, sq_mom=0.988, lr_sched="lin_cos_log_15", wd_sched="cos_log_15",
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act="silu", p_drop=0.05, p_drop_sched="expm4t",
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name="gelu", plr_lr_factor=0.1151, plr_sigma=2.33,
    ls_eps=0.01, ls_eps_sched="sqrt_cos", add_front_scale=False, bias_init_mode="neg-uniform-dynamic-2",
    tfms=["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding", "l2_normalize"])


def train_realmlp(with_te):
    from pytabkit import RealMLP_TD_Classifier
    Xc = full.copy()
    for c in CATS:
        Xc[c] = Xc[c].astype(str).astype("category")
    XA, XT = Xc.iloc[:len(tr)].reset_index(drop=True), Xc.iloc[len(tr):].reset_index(drop=True)
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if with_te:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], FEATS)
        m = RealMLP_TD_Classifier(device="cuda", random_state=SEED, val_metric_name="1-auc_ovr", verbosity=0, **REALMLP)
        m.fit(Xa, y[a], X_val=Xb, y_val=y[b], cat_col_names=CATS)
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


for name, with_te in (("realmlp_te", True), ("realmlp_v1", False)):
    if RETRAIN_NN:
        OOF[name], PRED[name] = train_realmlp(with_te)
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", "s6e11-fraud-series-artifacts"))[:len(tr)].astype(float)
        PRED[name] = np.load(find(f"{name}_test.npy", "s6e11-fraud-series-artifacts"))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

## 1.6 Part 1 stack

A logistic regression on the five logits, cross-validated on a second 5-fold split so the stacker never
scores rows it was fitted on. Part 1's submission is written to `submission_part1.csv`.

In [ ]:
P1_NAMES = ["lgb_te_l4", "glm_te", "realmlp_te", "glm_sb", "realmlp_v1"]
Lo = np.column_stack([logit(OOF[n]) for n in P1_NAMES])
Lp = np.column_stack([logit(PRED[n]) for n in P1_NAMES])
st = np.zeros(len(y))
for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(Lo, y):
    st[b] = LogisticRegression(C=1.0, max_iter=1000).fit(Lo[a], y[a]).decision_function(Lo[b])
stack = LogisticRegression(C=1.0, max_iter=1000).fit(Lo, y)
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P1_NAMES] + [roc_auc_score(y, st)],
                      "stack coefficient": list(stack.coef_[0]) + [np.nan]},
                     index=P1_NAMES + ["stack (cross-validated)"]).round({"CV AUC": 5, "stack coefficient": 3}))
pd.DataFrame({"id": te["id"], TARGET: 1 / (1 + np.exp(-stack.decision_function(Lp)))}).to_csv("submission_part1.csv", index=False)

## 1.7 What did not help in Part 1 (fold 0 unless noted)

| idea | result | note |
|---|---|---|
| original 20k rows appended to training | -0.0003 | the original rows are distributed differently |
| ratios (amount/balance, ...) | -0.0004 |  |
| number of flags set | -0.0001 | trees find it anyway |
| numeric columns as LightGBM categoricals | -0.0054 | per-value splits overfit; smoothed out-of-fold encodings are the way |
| pair target encodings (91 pairs) | -0.0006 | 4-leaf trees already model the pairs |
| value counts on top of the encodings | +0.00004 | noise |
| TabM | 0.9425 | vs RealMLP 0.9443 |
| GLM with pairwise products and category pairs | -0.0016 | overfits |
| CatBoost with every number as a categorical (ordered target statistics) | 0.9448 | vs 0.9474 for 4-leaf LightGBM + TE, and 30x slower |

---
# Part 2 — Two views of the data

*Public LB 0.94945 · 5-fold CV 0.94923 · second version of this notebook*

In [ ]:
draw_part2(
    stats=[('Public LB', '0.94945', '#1 · +0.00017 over Part 1'), ('5-fold CV AUC', '0.94923', '+0.00031 over Part 1'), ('Candidates', '12 new', '6 members in the final blend'), ('Best single model', '0.94885', '4-leaf LightGBM + TE, smoothing 100')],
    names=['lgb_te_s100', 'lgb_te_s300', 'xgb_te_d2', 'realmlp_te', 'lgb_raw', 'lgb_raw_cnt'],
    corr=[[1, 0.9959, 0.993, 0.9747, 0.9767, 0.9763], [0.9959, 1, 0.99, 0.9723, 0.9756, 0.9753], [0.993, 0.99, 1, 0.9772, 0.976, 0.9746], [0.9747, 0.9723, 0.9772, 1, 0.9659, 0.9639], [0.9767, 0.9756, 0.976, 0.9659, 1, 0.9912], [0.9763, 0.9753, 0.9746, 0.9639, 0.9912, 1]],
    gains=[('lgb_raw_l4', 'raw view', 1.43), ('lgb_raw_l4_cnt', 'raw view', 1.47), ('xgb_raw_d3', 'raw view', 1.12), ('lgb_raw_l31', 'raw view', 0.72), ('lgb_te_l4_s100', 'target-encoded view', 0.91), ('lgb_te_l4_s300', 'target-encoded view', 1.0), ('lgb_te_l4_s5', 'target-encoded view', 0.35), ('xgb_te_d2', 'target-encoded view', 0.64), ('realmlp_te_s2', 'neural net', 0.21), ('realmlp_te_cnt', 'neural net', 0.23), ('realmlp_cnt', 'neural net', 0.54), ('realmlp_allcat', 'neural net', -0.2)],
    smoothing={'weights': [5, 20, 100, 300], 'cv': [0.948777, 0.948845, 0.948852, 0.94873], 'kept': [100, 300]},
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.949229, 0.94945)],
    blend=[('LightGBM 4 leaves + TE (smoothing 100)', 'target-encoded view', 0.25), ('XGBoost depth 2 + TE', 'target-encoded view', 0.25), ('LightGBM 4 leaves + TE (smoothing 300)', 'target-encoded view', 0.125), ('LightGBM 4 leaves, raw columns', 'raw view', 0.125), ('LightGBM 4 leaves, raw + value counts', 'raw view', 0.125), ('RealMLP + TE', 'neural net', 0.125)],
    title="S6E11 Credit Card Fraud  ·  Part 2",
    subtitle="Two views of the same data: target-encoded and raw shallow trees, blended by rank hill climbing")
plt.show()

Part 1's members all look at the data through the target encodings. Part 2 asks what to put *next to* them.
The **raw view** is the 24 columns as they are, optionally with value counts (how often each value occurs
in train + test + original). Both views share the folds.

## 2.1 Five more tree members

All shallow: LightGBM with 4 leaves (target-encoded with smoothing 100 and 300, raw, raw + counts) and
XGBoost with depth 2 on the target-encoded view.

In [ ]:
CNT = pd.DataFrame({f"cnt_{c}": full[c].map(allv[c].value_counts()).astype("float32") for c in NUMS})
Ctr, Cte = CNT.iloc[:len(tr)].reset_index(drop=True), CNT.iloc[len(tr):].reset_index(drop=True)
XGB = dict(n_estimators=20000, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.5,
           min_child_weight=5, reg_lambda=1.0, max_bin=256, tree_method="hist", n_jobs=-1, eval_metric="auc",
           early_stopping_rounds=300, enable_categorical=True, max_cat_to_onehot=1)

run_lgb("lgb_te_l4_s100", Xtr, Xte, use_te=True, smooth=100.0)
run_lgb("lgb_te_l4_s300", Xtr, Xte, use_te=True, smooth=300.0)
run_lgb("lgb_raw_l4", Xtr, Xte, use_te=False)
run_lgb("lgb_raw_l4_cnt", pd.concat([Xtr, Ctr], axis=1), pd.concat([Xte, Cte], axis=1), use_te=False)

oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
for f, (a, b) in enumerate(FOLDS):
    Xa, (Xb, Xt) = add_te(Xtr.iloc[a], y[a], [Xtr.iloc[b], Xte], FEATS)
    m = xgb.XGBClassifier(**XGB, random_state=SEED)
    m.fit(Xa, y[a], eval_set=[(Xb, y[b])], verbose=False)
    oof[b] = m.predict_proba(Xb)[:, 1]
    pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
    print(f"xgb_te_d2 fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {m.best_iteration} trees ({time.time() - t0:.0f}s)")
OOF["xgb_te_d2"], PRED["xgb_te_d2"] = oof, pred
print(f"==> xgb_te_d2: CV AUC {roc_auc_score(y, oof):.5f}")

## 2.2 Two views, two blocks

The target-encoded trees agree with each other at 0.99+, so a second or third one adds little. The raw-view
trees sit at about 0.976 from them, which is far enough to matter, and RealMLP is further still.

In [ ]:
P2_NAMES = ["lgb_te_l4_s100", "lgb_te_l4_s300", "xgb_te_d2", "realmlp_te", "lgb_raw_l4", "lgb_raw_l4_cnt"]
display(pd.DataFrame({n: OOF[n] for n in P2_NAMES}).corr(method="spearman").round(4))

What each candidate adds, one at a time on top of Part 1's five-model stack (CV 0.94892; cross-validated
stack AUC, × 1e-4), all on the same folds:

| candidate | view | own CV | gain over Part 1 |
|---|---|---|---|
| `lgb_raw_l4_cnt` | raw | 0.94778 | +1.47 |
| `lgb_raw_l4` | raw | 0.94777 | +1.43 |
| `xgb_raw_d3` | raw | 0.94749 | +1.12 |
| `lgb_te_l4_s300` | target-encoded | 0.94873 | +1.00 |
| `lgb_te_l4_s100` | target-encoded | 0.94885 | +0.91 |
| `lgb_raw_l31` | raw | 0.94706 | +0.72 |
| `xgb_te_d2` | target-encoded | 0.94881 | +0.64 |
| `realmlp_cnt` | neural net, raw + counts | 0.94664 | +0.54 |
| `lgb_te_l4_s5` | target-encoded | 0.94878 | +0.35 |
| `realmlp_te_cnt` | neural net | 0.94760 | +0.23 |
| `realmlp_te_s2` | neural net | 0.94762 | +0.21 |
| `realmlp_allcat` | neural net, per-value embeddings | 0.94640 | -0.20 |

The raw-view trees lead, although each of them is weaker than any target-encoded tree on its own.
More neural variants (other seeds, counts, per-value embeddings) add almost nothing.

## 2.3 Rank hill climbing

AUC only depends on the order of the predictions, so the members become ranks (scaled to [0, 1]) and are
added greedily: at each step, add the member (repeats allowed) that raises the out-of-fold AUC most; stop
when nothing helps. The weights come out in steps of 1/8 with six members, so there is little room to
overfit. On all 13 candidates the same procedure picks exactly these six. Part 2's submission is written
to `submission_part2.csv`.

In [ ]:
def hill_climb(R, names, max_iter=60):
    counts, cur, best = np.zeros(R.shape[1]), np.zeros(len(y)), 0.0
    for _ in range(max_iter):
        scores = [roc_auc_score(y, cur + R[:, j]) for j in range(R.shape[1])]
        j = int(np.argmax(scores))
        if scores[j] <= best + 1e-7:
            break
        best, cur = scores[j], cur + R[:, j]
        counts[j] += 1
        print(f"  add {names[j]:16s} -> OOF AUC {best:.6f}")
    return counts / counts.sum(), best


Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P2_NAMES])
Rp = np.column_stack([rankdata(PRED[n]) / len(te) for n in P2_NAMES])
w, auc = hill_climb(Ro, P2_NAMES)
print(f"\nPart 2 blend OOF AUC {auc:.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P2_NAMES], "weight": w}, index=P2_NAMES).round(5))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part2.csv", index=False)
sub.head()

## 2.4 What did not help in Part 2

| idea | result | note |
|---|---|---|
| TE smoothing 5 as an extra member | +0.35 alone, -0.08 in the final set | too close to smoothing 20 |
| XGBoost + TE, depth 3 / 4 | fold 0: 0.94693 / 0.94686 | depth 2: 0.94726 |
| CatBoost with every number as a categorical (GPU) | fold 0: 0.9448 | 32 min per fold |
| CatBoost on the raw columns, depth 6 (GPU) | fold 0: 0.9426 | 17 min per fold, stopped |
| RealMLP with per-value embeddings | CV 0.94640, -0.20e-4 in the stack |  |
| pair target encodings (91 pairs) | fold 0: -0.0006 | 4-leaf trees already model pairs |
| logistic stack over all 13 candidates | CV 0.94920 | hill climbing over the same 13: 0.94923 |

---
# Part 3 — The digits carry the label

*Public LB 0.95113 · 5-fold CV 0.95073*

Parts 1 and 2 treated every number as a value: its size, or the fraud rate of rows with exactly that value.
Part 3 looks at how the numbers are *written*. The synthetic rows were made by a generator trained on the
original data, and if it writes a number digit by digit, the digits themselves can carry the label.

In [ ]:
draw_part3(
    stats=[('Public LB', '0.95113', '+0.00168 over Part 2'), ('5-fold CV AUC', '0.95073', '+0.00150 over Part 2'), ('Best single model', '0.95040', '4-leaf LightGBM + digit keys'), ('New keys', '4 per number', 'cents · int(x) · int(x/10) · int(10x)')],
    digits={'cols': ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km', 'account_balance_usd'], 'rate': [[0.01904, 0.01499, 0.01563, 0.01542, 0.01925, 0.01772, 0.01478, 0.01729, 0.01712, 0.01633], [0.02204, 0.01584, 0.01603, 0.0139, 0.01919, 0.0185, 0.01265, 0.01491, 0.01626, 0.01679], [0.01942, 0.01621, 0.01493, 0.01505, 0.01813, 0.01727, 0.01482, 0.0173, 0.01675, 0.01677], [0.01752, 0.01647, 0.01557, 0.01632, 0.01678, 0.01784, 0.01584, 0.01639, 0.01743, 0.01686]], 'base': 0.0167},
    steps=[('per-value TE', 0.947461), ('+ cents', 0.947794), ('+ int(x), int(x/10)', 0.949663), ('+ int(10x)', 0.949783)],
    families=[('LightGBM, 4 leaves', 0.9488452589277478, 0.9503587999739997), ('XGBoost, depth 2', 0.9488134428126807, 0.950254313009181), ('EBM', 0.948771, 0.9498084549664615), ('GLM', 0.9470568398079565, 0.9487861923393974), ('RealMLP', 0.9477104886524447, 0.9485288055034796)],
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.94923, 0.94945), ('Part 3', 0.95073, 0.95113)],
    blend=[('lgb_diglead_l4', 'target-encoded view', 0.3333), ('lgb_dig_s100', 'target-encoded view', 0.3333), ('ebm_dig', 'target-encoded view', 0.1667), ('ebm_b4', 'raw view', 0.1667)],
    title="S6E11 Credit Card Fraud  ·  Part 3",
    subtitle="The synthetic generator writes numbers digit by digit, and the digits carry the label")
plt.show()

## 3.1 A fingerprint of the generator

Fraud rate by the first digit after the decimal point, for the four float columns with two decimals.
A physical quantity should not care about its first decimal, yet the same digits stand out in every column:
`.0` and `.4` above the base rate, `.2`, `.3` and `.6` below it.

In [ ]:
D4 = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]


def by_digit(df, col, digit):
    k = np.round(df[col].values * 100).astype(np.int64)        # the number in cents
    return pd.Series(df[TARGET].values).groupby(digit(k)).mean().reindex(range(10))


first_decimal = pd.DataFrame({c: by_digit(tr, c, lambda k: (k // 10) % 10) for c in D4}).T
display((first_decimal * 100).round(2).rename(columns=lambda d: f".{d}"))
_style()
fig = plt.figure(figsize=(11, 3.6))
_digits(fig, [0.2, 0.14, 0.62, 0.62], dict(cols=D4, rate=first_decimal.values.tolist(), base=y.mean()))
plt.show()

The integer part shows the same kind of pattern: a tens digit of 4 is above the base rate in `amount_usd`,
`hours_since_last_txn` and `account_balance_usd`, a 3 below it. The digit frequencies tell the same story
from the other side: train and test agree to within 0.2 percentage points, the original 20k rows differ by
up to 2.4. The generator left its handwriting on both train and test, so whatever the digits say about the
label in train should hold on the test rows.

In [ ]:
tens = pd.DataFrame({c: by_digit(tr, c, lambda k: (k // 1000) % 10) for c in D4}).T
display((tens * 100).round(2).rename(columns=lambda d: f"{d}_"))

FLOATS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd",
          "velocity_score", "merchant_risk_score"]


def digit_share(df, col):
    k = np.round(df[col].values * 100).astype(np.int64)
    return pd.Series((k // 10) % 10).value_counts(normalize=True).sort_index()


gap = pd.DataFrame({c: [(digit_share(tr, c) - digit_share(te, c)).abs().max(),
                        (digit_share(tr, c) - digit_share(og, c)).abs().max()] for c in FLOATS},
                   index=["train vs test", "train vs original"]).T * 100
display(gap.round(2).rename(columns=lambda s: s + " (max gap, pp)"))

## 3.2 Four keys per number

For each of the six float columns we add four keys and target-encode each one exactly like the values in
Part 1 (out of fold, smoothing 20):

| key | example for 1371.37 | what it groups |
|---|---|---|
| `cents` | 37 | the two decimals (also kept as a plain number) |
| `int(x)` | 1371 | the integer part |
| `int(x/10)` | 137 | the integer part without its last digit |
| `int(10x)` | 13713 | the value cut to one decimal |

The keys themselves are dropped after encoding; only their encodings (and `cents`) reach the model. Each
step on one 4-leaf LightGBM, fold 0 (the Part 2 features score 0.94746 there):

| features | fold-0 AUC | gain |
|---|---|---|
| Part 2 features (per-value TE of every column) | 0.94746 | — |
| + `cents` | 0.94779 | +3.3e-4 |
| + `int(x)`, `int(x/10)` (no `cents`) | 0.94893 | +14.6e-4 |
| + `cents`, `int(x)`, `int(x/10)` | 0.94966 | +22.0e-4 |
| + `cents`, `int(x)`, `int(x/10)`, `int(10x)` | 0.94978 | **+23.2e-4** |
| … + leading one and two digits of `int(x)` | 0.94988 | +24.2e-4 |

Finer histogram bins (`max_bin=4095`) on the old features give nothing (0.94751): the signal is not a
resolution problem but the digits themselves.

In [ ]:
def digit_frame(df):
    out = {}
    for c in FLOATS:
        out[f"cents_{c}"] = (np.round(df[c] * 100) % 100).astype("int16").values
    for c in FLOATS:
        out[f"key_int_{c}"] = np.floor(df[c]).astype("int64").values
        out[f"key_ten_{c}"] = np.floor(df[c] / 10).astype("int64").values
    for c in FLOATS:
        out[f"key_dec_{c}"] = np.floor(df[c] * 10 + 1e-6).astype("int64").values
    return pd.DataFrame(out)


DG = digit_frame(full)
CENTS = [c for c in DG if c.startswith("cents_")]
KEYS = [c for c in DG if c.startswith("key_")]
XDtr = pd.concat([Xtr, DG.iloc[:len(tr)].reset_index(drop=True)], axis=1)
XDte = pd.concat([Xte, DG.iloc[len(tr):].reset_index(drop=True)], axis=1)
DIG_TE = FEATS + CENTS + KEYS          # every column, the cents and the three rounded keys


def drop_keys(D):
    return D[[c for c in D.columns if not c.startswith("key_")]]


def run_tree(name, kind, XA, XT, te_cols, smooth=20.0, seed=SEED, teseed=0, **over):
    oof, pred, its, t0 = np.zeros(len(tr)), np.zeros(len(te)), [], time.time()
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if te_cols:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], te_cols, smooth, seed=teseed)
        Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        if kind == "lgb":
            m = lgb.LGBMClassifier(**{**LGB, **over}, random_state=seed)
            m.fit(Xa, y[a], eval_set=[(Xb, y[b])], eval_metric="auc", callbacks=[lgb.early_stopping(300, verbose=False)])
            it = m.best_iteration_
        else:
            m = xgb.XGBClassifier(**{**XGB, **over}, random_state=seed)
            m.fit(Xa, y[a], eval_set=[(Xb, y[b])], verbose=False)
            it = m.best_iteration
        oof[b] = m.predict_proba(Xb)[:, 1]
        pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
        its.append(it)
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {it} trees ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name], ITS[name] = oof, pred, its
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")


run_tree("lgb_dig_l4", "lgb", XDtr, XDte, DIG_TE)

## 3.3 Every family, retrained with the digits

The same keys for the other members: the 4-leaf LightGBM with the leading one and two digits of the
integer part as two more keys (the best single model, CV 0.95040), the same model with TE smoothing 100
as a second view, and a depth-2 XGBoost. Two more views were measured and left out because the blend did
not pick them (section 3.5).

In [ ]:
# the leading one and two digits of the integer part, with its length (1371.37 -> 41 and 413)
def lead_frame(df):
    out = {}
    for c in FLOATS:
        i = np.floor(df[c]).astype("int64").astype(str)
        out[f"key_l1_{c}"] = (i.str.len() * 10 + i.str[0].astype(int)).values
        out[f"key_l2_{c}"] = (i.str.len() * 100 + i.str[:2].astype(int)).values
    return pd.DataFrame(out)


LD = lead_frame(full)
XLtr = pd.concat([XDtr, LD.iloc[:len(tr)].reset_index(drop=True)], axis=1)
XLte = pd.concat([XDte, LD.iloc[len(tr):].reset_index(drop=True)], axis=1)
DEC = [c for c in KEYS if c.startswith("key_dec_")]
LEAD_TE = FEATS + CENTS + [c for c in KEYS if c not in DEC] + list(LD.columns) + DEC

run_tree("lgb_diglead_l4", "lgb", XLtr, XLte, LEAD_TE)              # + the leading digits
run_tree("lgb_dig_s100", "lgb", XDtr, XDte, DIG_TE, smooth=100.0)   # smoothing as a second view
run_tree("xgb_dig_d2", "xgb", XDtr, XDte, DIG_TE)                     # depth-2 XGBoost

A GLM with the encodings of all the keys, and the EBM / RealMLP members. The RealMLP members were trained
on our GPU and the EBMs on 16 CPU cores (about 2 hours each), so their out-of-fold and test predictions are
loaded from our dataset; the training code is here (`RETRAIN_NN`, `RETRAIN_EBM`).

In [ ]:
run_glm("glm_dig", max_card=1000, C=0.3, with_te=True, src=(XDtr, XDte), te_cols=DIG_TE)

RETRAIN_EBM = False


def train_ebm(dig, outer_bags=4):
    from interpret.glassbox import ExplainableBoostingClassifier
    XA, XT = (XDtr, XDte) if dig else (Xtr, Xte)
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if dig:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], DIG_TE)
            Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        m = ExplainableBoostingClassifier(interactions=10, max_bins=256, outer_bags=outer_bags, n_jobs=-1,
                                          random_state=42).fit(Xa, y[a])
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


def train_realmlp_dig(seed=SEED, smooth=20.0):
    from pytabkit import RealMLP_TD_Classifier
    Xc = XDtr.copy(), XDte.copy()
    for X in Xc:
        for c in CATS:
            X[c] = X[c].astype(str).astype("category")
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, (Xb, Xt) = add_te(Xc[0].iloc[a], y[a], [Xc[0].iloc[b], Xc[1]], DIG_TE, smooth)
        Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        m = RealMLP_TD_Classifier(device="cuda", random_state=seed, val_metric_name="1-auc_ovr", verbosity=0, **REALMLP)
        m.fit(Xa, y[a], X_val=Xb, y_val=y[b], cat_col_names=CATS)
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


for name in ['realmlp_dig', 'ebm_dig', 'ebm_b4']:
    if RETRAIN_EBM and name.startswith("ebm"):
        OOF[name], PRED[name] = train_ebm(dig="dig" in name)
    elif RETRAIN_NN and name.startswith("realmlp"):
        OOF[name], PRED[name] = train_realmlp_dig()
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", "s6e11-fraud-series-artifacts"))[:len(tr)].astype(float)
        PRED[name] = np.load(find(f"{name}_test.npy", "s6e11-fraud-series-artifacts"))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

| family | Part 2 features | + digit keys | gain |
|---|---|---|---|
| LightGBM, 4 leaves | 0.94885 | 0.95036 | +15e-4 |
| XGBoost, depth 2 | 0.94881 | 0.95025 | +14e-4 |
| EBM | 0.94877 | 0.94981 | +10e-4 |
| GLM | 0.94706 | 0.94879 | +17e-4 |
| RealMLP | 0.94771 | 0.94853 | +8e-4 |

## 3.4 The Part 3 blend

The same rank hill climbing as in Part 2, over the digit-aware members and everything from Parts 1 and 2.
Out of 18 candidates it keeps 4, in steps of 1/6: `lgb_diglead_l4` 2/6, `lgb_dig_s100` 2/6, `ebm_dig` 1/6, `ebm_b4` 1/6 (OOF AUC 0.95073). None of the Part 1 and Part 2 members is picked except the raw-view EBM, which looks at the data most differently. Part 3's submission is written to `submission_part3.csv`.

In [ ]:
P3_NAMES = ['lgb_dig_l4', 'lgb_diglead_l4', 'lgb_dig_s100', 'xgb_dig_d2', 'glm_dig', 'realmlp_dig', 'ebm_dig', 'ebm_b4', 'lgb_te_l4', 'glm_te', 'realmlp_te', 'glm_sb', 'realmlp_v1', 'lgb_te_l4_s100', 'lgb_te_l4_s300', 'xgb_te_d2', 'lgb_raw_l4', 'lgb_raw_l4_cnt']
Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P3_NAMES])
Rp = np.column_stack([rankdata(PRED[n]) / len(te) for n in P3_NAMES])
w, auc = hill_climb(Ro, P3_NAMES)
print(f"\nPart 3 blend OOF AUC {auc:.6f}")
keep = w > 0
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P3_NAMES], "weight": w},
                     index=P3_NAMES)[keep].round(5))
display(pd.DataFrame({n: OOF[n] for n in np.array(P3_NAMES)[keep]}).corr(method="spearman").round(4))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part3.csv", index=False)
sub.head()

## 3.5 What did not help in Part 3

| idea | result | note |
|---|---|---|
| Per-value encoding of the blend's residuals | no gain | only negative weights help (a leak inside the OOF): the values are used up |
| First and second decimal as two separate keys | −0.00003 (fold 0) | `cents` already holds both |
| Ones and tens digit of the integer part | −0.00015 | `int(x)` and `int(x/10)` cover them |
| `int(x/100)` | ±0.00000 |  |
| Pairs of first decimals across columns (6 keys) | −0.00005 | the digit effects add up column by column |
| Finer LightGBM histograms (`max_bin=4095`), no keys | +0.00005 | not a resolution problem |
| TE smoothing 10 / 100 for the digit model | −0.00036 / −0.00038 | 20 stays; 100 still joins the blend as a second view |
| 7 leaves instead of 4 | CV 0.95016 | shallow still wins |
| Only the rounded keys encoded, no per-value TE (`lgb_dk_l4`) | CV 0.94901 | a different view, but not picked by the blend |
| Raw columns + decimals as numbers, no encodings (`lgb_rawdig_l4`) | CV 0.94838 | +0.0006 over the raw view of Part 2, not picked: the raw EBM is the better raw view |
| RealMLP on raw columns + decimals (no encodings) | CV 0.94615 | the network needs the encodings |
| Label correlation along `id` | |r| < 0.002 | no batch structure in the generated rows |

---
# Part 4 — Same models, more data

*Public LB 0.95154 · 5-fold CV 0.95079 · the summary figure is at the top of the notebook*

Part 4 finds no new signal. It estimates Part 3's signal with less noise, in three ways: one more model per
member trained on every training row, the mean of several seeds, and a GLM that fits one coefficient per
value for all values at once.

In [ ]:
draw_part4(
    stats=[('Public LB', '0.95154', '+0.00041 over Part 3'), ('5-fold CV AUC', '0.95079', '+0.00006 over Part 3'), ('Refit on all rows', '+0.00019', 'public LB, same models and weights'), ('Joint GLM', '0.94991', "+0.00112 over Part 3's GLM")],
    steps=[('Part 3 blend, 5-fold average', 0.95113), ('same blend, refit on all rows', 0.95132), ('+ seed bags, joint GLM', 0.95154)],
    csweep={'C': [0.003, 0.005, 0.01, 0.03, 0.1, 0.3], 'series': {'one-hot only': [0.9432, None, 0.946641, 0.947582, 0.946357, 0.942588], 'one-hot + TE logits': [0.949056, 0.949178, 0.94915, 0.948535, 0.946593, None]}, 'ref': ['TE logits only\n(Part 3 GLM)', 0.947748]},
    families=[('LightGBM + leading digits', 0.9503999740916812, 0.9505103584240805), ('LightGBM, smoothing 100', 0.9502855033473656, 0.9504258336734523), ('GLM', 0.9487861923393974, 0.9499073661219074)],
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.94923, 0.94945), ('Part 3', 0.95073, 0.95113), ('Part 4', 0.95079, 0.95154)],
    blend=[('lgb_diglead_bag', 'target-encoded view', 0.3333), ('glm_ohk_c01', 'joint GLM', 0.2222), ('lgb_dig_s100_bag', 'target-encoded view', 0.1111), ('ebm_dig', 'target-encoded view', 0.1111), ('lgb_rawdig_l4', 'raw view', 0.1111), ('lgb_dig_l7', 'target-encoded view', 0.1111)],
    title="S6E11 Credit Card Fraud  ·  Part 4",
    subtitle="Same models, more data: refit on every row, bag the seeds, and fit every value jointly in a GLM")
plt.show()

## 4.1 Refit on every training row

Every test prediction so far was the average of the five fold models, each trained on 80% of the training
rows. With per-value target encodings, 80% of the rows also means noisier encodings (in
`account_balance_usd` the median value has 3 rows). So each member gets one more model, trained on all
training rows:

- the encodings are fitted on all training rows (inner folds for the training rows themselves, as before),
- the number of trees is 1.25 x the mean best iteration of the five folds (25% more rows), no early stopping.

The out-of-fold predictions, and with them the CV and the blend weights, do not change; only the test
predictions do. That alone moved the Part 3 blend from LB 0.95113 to **0.95132**. Here is the refit of
Part 3's `lgb_diglead_l4` next to its 5-fold average (rank correlation 0.9955), and checked
against the refit stored in our dataset.

In [ ]:
ART = "s6e11-fraud-series-artifacts"


def refit_full(kind, XA, XT, te_cols, n_trees, smooth=20.0, seed=SEED, teseed=0, **over):
    """One model on every training row: encodings fitted on all of train, n_trees trees, no early stopping."""
    Xa, Xt = XA, XT
    if te_cols:
        Xa, (Xt,) = add_te(XA, y, [XT], te_cols, smooth, seed=teseed)
    Xa, Xt = drop_keys(Xa), drop_keys(Xt)
    if kind == "lgb":
        m = lgb.LGBMClassifier(**{**LGB, **over, "n_estimators": n_trees}, random_state=seed)
    else:
        params = {k: v for k, v in {**XGB, **over}.items() if k != "early_stopping_rounds"}
        m = xgb.XGBClassifier(**{**params, "n_estimators": n_trees}, random_state=seed)
    return m.fit(Xa, y).predict_proba(Xt)[:, 1]


n_trees = int(round(np.mean(ITS["lgb_diglead_l4"]) * 1.25))
FULL["lgb_diglead_l4"] = refit_full("lgb", XLtr, XLte, LEAD_TE, n_trees)
saved = np.load(find("lgb_diglead_l4_full.npy", ART))[:len(te)]
print(f"lgb_diglead_l4: the folds stopped at {ITS['lgb_diglead_l4']} trees -> refit with {n_trees}")
print(f"rank correlation, refit vs 5-fold average: {spearmanr(FULL['lgb_diglead_l4'], PRED['lgb_diglead_l4']).correlation:.4f}")
print(f"largest difference to the refit in our dataset: {np.abs(FULL['lgb_diglead_l4'] - saved).max():.1e}")

## 4.2 Bag the seeds

Two things are random in a tree member: the model seed (row and column subsampling) and the inner folds of
the target encoding. Changing both moves a 4-leaf LightGBM by up to 0.0002 CV, as much as many feature
ideas. The mean of three seeds is steadier than any one of them (5-fold CV AUC, probabilities averaged):

| member | seed 42, TE folds 0 | seed 1, TE folds 1 | seed 2, TE folds 2 | mean of the three |
|---|---|---|---|---|
| `lgb_diglead_l4` (Part 3) | 0.95040 | 0.95020 | 0.95023 | **0.95051** |
| `lgb_dig_s100` (Part 3) | 0.95029 | 0.95021 | 0.95014 | **0.95043** |

Five seeds gave 0.95054 for the first one: most of the gain is in the first three. Seed 42 happened to be
the lucky one (0.95040, the other four 0.95020-0.95035), a reminder of how much a single run can vary.
The bags are trained with the code in 4.4 (`RETRAIN_P4`).

## 4.3 A GLM that fits every value jointly

Part 3's GLM sees each value through its target encoding: the fraud rate of rows with that value, column by
column. Each encoding is marginal (it knows nothing about the other columns), and a rare value's encoding is
shrunk towards the global fraud rate.

Part 4's GLM adds a one-hot column for every exact value of every numeric column and for every digit key of
the float columns (values seen fewer than 5 times share a column), next to the encoding logits, and fits all
coefficients together under one L2 penalty. The coefficients then correct the encodings for each other, and
because a value's effect is the sum of its exact-value, `int(10x)`, `int(x)` and `int(x/10)` coefficients, a
rare value falls back to its coarser keys instead of to the global rate. The same idea as a tree feature
(encodings shrunk towards the parent key) did not help (4.5): it works here because the fit is joint.

| GLM, fold 0 | AUC |
|---|---|
| Part 3 GLM: splines + bins + TE logits of every key, C 0.3 | 0.94775 |
| one-hot of every value and key + splines, no TE, best C (0.03) | 0.94758 |
| both: TE logits + one-hot, C 0.03 | 0.94854 |
| both, C 0.01 (`glm_ohk_c01`, the Part 4 member) | 0.94915 |
| both, C 0.005 | 0.94918 |
| … + leading digits, more key levels, TE smoothing 50 (measured later) | 0.94941 |

With tens of thousands of one-hot columns it needs a much stronger penalty than Part 3's GLM (`C=0.01`
instead of 0.3). It trains in minutes, so it runs here; the blend below uses the predictions stored in our
dataset so that this notebook reproduces the submitted file exactly.

In [ ]:
def key_onehot(min_freq=5):
    """One-hot of every exact numeric value and of the digit keys of the float columns."""
    keys = {f"v_{c}": full[c] for c in NUMS}
    for c in FLOATS:
        i = np.floor(full[c]).astype("int64")
        keys.update({f"cents_{c}": np.round(full[c] * 100) % 100, f"int_{c}": i, f"ten_{c}": i // 10,
                     f"dec_{c}": np.floor(full[c] * 10 + 1e-6),
                     f"l2_{c}": i.astype(str).str.len() * 100 + i.astype(str).str[:2].astype(int)})
    K = pd.DataFrame({k: np.asarray(v, dtype=float) for k, v in keys.items()})
    return OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq).fit_transform(K)


KO = key_onehot()
print(f"{KO.shape[1]:,} one-hot columns for values and digit keys\n")
run_glm("glm_ohk_c01", max_card=1000, C=0.01, with_te=True, src=(XDtr, XDte), te_cols=DIG_TE, extra=KO, refit=True)
for kind, arr, n in (("OOF", OOF, len(tr)), ("refit test", FULL, len(te))):
    saved = np.load(find(f"glm_ohk_c01_{'oof' if kind == 'OOF' else 'full'}.npy", ART))[:n]
    print(f"glm_ohk_c01 {kind}: largest difference to our dataset {np.abs(arr['glm_ohk_c01'] - saved).max():.1e}")

## 4.4 The Part 4 blend

The other members were trained outside this notebook (the three-seed bags with their refits take several
hours on 4 CPU cores); their out-of-fold and refit test predictions are loaded from our dataset, and the code
that made them is here (`RETRAIN_P4 = True` retrains them). Then the same rank hill climbing as before:
weights from the out-of-fold predictions, test side from the refits. Out of eleven candidates (also the
`C=0.03` GLM, the raw EBM, the depth-2 XGBoost, RealMLP and the keys-only LightGBM) it keeps these six. This
is Part 4's submission, written to `submission_part4.csv`.

In [ ]:
RETRAIN_P4 = False                      # True: retrain the seed bags and every refit here (several hours)
SEEDS = [(SEED, 0), (1, 1), (2, 2)]     # (model seed, seed of the encoding's inner folds)


def bag_trees(name, kind, XA, XT, te_cols, smooth=20.0, **over):
    """Three seeds of one tree member: mean out-of-fold prediction and mean refit test prediction."""
    oofs, fulls = [], []
    for s, ts in SEEDS:
        run_tree(f"{name}_seed{s}", kind, XA, XT, te_cols, smooth, seed=s, teseed=ts, **over)
        n = int(round(np.mean(ITS[f"{name}_seed{s}"]) * 1.25))
        oofs.append(OOF[f"{name}_seed{s}"])
        fulls.append(refit_full(kind, XA, XT, te_cols, n, smooth, seed=s, teseed=ts, **over))
    return np.mean(oofs, axis=0), np.mean(fulls, axis=0)


def tree_with_refit(name, kind, XA, XT, te_cols, **over):
    run_tree(name, kind, XA, XT, te_cols, **over)
    return OOF[name], refit_full(kind, XA, XT, te_cols, int(round(np.mean(ITS[name]) * 1.25)), **over)


def raw_digit_frames():
    """Raw columns + cents and the first / second decimal as plain numbers (no encodings)."""
    k = {c: np.round(full[c] * 100).astype("int64").values for c in FLOATS}
    RD = pd.concat([DG[CENTS], pd.DataFrame({n: v for c in FLOATS for n, v in
                    ((f"d1_{c}", ((k[c] // 10) % 10).astype("int8")), (f"d2_{c}", (k[c] % 10).astype("int8")))})], axis=1)
    return (pd.concat([Xtr, RD.iloc[:len(tr)].reset_index(drop=True)], axis=1),
            pd.concat([Xte, RD.iloc[len(tr):].reset_index(drop=True)], axis=1))


def ebm_refit():
    from interpret.glassbox import ExplainableBoostingClassifier
    Xa, (Xt,) = add_te(XDtr, y, [XDte], DIG_TE)
    m = ExplainableBoostingClassifier(interactions=10, max_bins=256, outer_bags=4, n_jobs=-1, random_state=42)
    return m.fit(drop_keys(Xa), y).predict_proba(drop_keys(Xt))[:, 1]


TRAIN_P4 = {
    "lgb_diglead_bag": lambda: bag_trees("lgb_diglead", "lgb", XLtr, XLte, LEAD_TE),
    "lgb_dig_s100_bag": lambda: bag_trees("lgb_dig_s100", "lgb", XDtr, XDte, DIG_TE, smooth=100.0),
    "glm_ohk_c01": lambda: (OOF["glm_ohk_c01"], FULL["glm_ohk_c01"]),          # trained live in 4.3
    "ebm_dig": lambda: (OOF["ebm_dig"], ebm_refit()),                          # OOF from Part 3
    "lgb_rawdig_l4": lambda: tree_with_refit("lgb_rawdig_l4", "lgb", *raw_digit_frames(), []),
    "lgb_dig_l7": lambda: tree_with_refit("lgb_dig_l7", "lgb", XDtr, XDte, DIG_TE, num_leaves=7),
}
P4_NAMES = ['lgb_diglead_bag', 'lgb_dig_s100_bag', 'glm_ohk_c01', 'ebm_dig', 'lgb_rawdig_l4', 'lgb_dig_l7']
for name in P4_NAMES:
    if RETRAIN_P4:
        OOF[name], FULL[name] = TRAIN_P4[name]()
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", ART))[:len(tr)].astype(float)
        FULL[name] = np.load(find(f"{name}_full.npy", ART))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P4_NAMES])
Rp = np.column_stack([rankdata(FULL[n]) / len(te) for n in P4_NAMES])
w, auc = hill_climb(Ro, P4_NAMES)
print(f"\nPart 4 blend OOF AUC {auc:.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P4_NAMES], "weight": w}, index=P4_NAMES).round(5))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part4.csv", index=False)
sub.head()

## 4.5 What the public board can and cannot tell

After this submission we kept going: more seeds, a GLM with more key levels (CV 0.95012), and EBMs with the
leading digits and 25 interactions (CV 0.95026, the best EBM so far). The CV kept rising, the public score
did not:

| submission | what changed | 5-fold CV | public LB |
|---|---|---|---|
| Part 4 | | 0.95079 | 0.95154 |
| `ens_v5_full` | 5 seeds per tree bag, improved GLM (CV 0.95012), 2 EBM seeds | 0.95085 | 0.95151 |
| `ens_v7_full` | fixed weights, EBM with leading digits and 25 interactions (CV 0.95026) | 0.95088 | 0.95145 |

The differences are 0.0001, inside the noise of a public board that scores 20% of the test rows (we
estimate the noise of the difference between two close submissions at about ±0.00014). The CV itself is optimistic for tuned
blends; scored honestly (weights chosen on four folds, scored on the fifth), the hill-climbing gain shrinks:

| blend weights | mean fold AUC |
|---|---|
| hill climbing, fitted on all OOF rows (in-sample) | 0.950843 |
| hill climbing, fitted on 4 folds, scored on the 5th | 0.950785 |
| weight grid, chosen on 4 folds, scored on the 5th | 0.950804 |
| equal weights on the 4 strongest bags | 0.950803 |

Simple equal weights on the four strongest bags score as well as the tuned weights. From here, the final
choice should follow honest CV rather than the public board.

## 4.6 What did not help in Part 4

| idea | result | note |
|---|---|---|
| GPT-2 tokens of the integer part as keys (`1269` → `12`·`69`) | −0.00016 (fold 0) | the digit keys already cover it |
| Hierarchical TE for the trees (rare values shrunk to their int(10x), int(x) parent) | −0.0003 (fold 0) | works in the GLM (jointly fitted), not as a tree feature |
| Inner folds of the TE: 10 / 20 instead of 5 | −0.0006 (fold 0) | the noise of 5 inner folds regularizes |
| LightGBM: lr 0.015, 3 leaves, colsample 0.3 / 0.8, path_smooth, min_child 50 / 200 / 300 | −0.00003 to −0.0003 | Part 3 settings are a local optimum |
| CatBoost on GPU with the digit keys as categoricals | 0.94603 (fold 0) |  |
| Factorization machine over per-value embeddings (pairwise terms) | 0.937–0.946 (fold 0) | pairs of values are noise: the signal is additive per column |
| DeepFM / embedding MLP | 0.9488 (fold 0) | RealMLP level |
| LightGBM boosted from the GLM's logit, or with it as an input | 0.94950 / 0.94921 (fold 0) | below the plain LightGBM; combine by rank blending only |
| Is the value in the original data? Value counts, velocity residual | no residual signal | screened on top of the best member |
| Digit key × category pairs (96) | max +0.00006 | noise |

---
# Part 5 — Three leaves, a plain GLM, and a public board that disagrees

*Public LB 0.95160 · 5-fold CV 0.95092 · the summary figure is at the top of the notebook*

Part 5 started from the public notebooks posted after Part 4 (credits at the top). Two of their ideas hold up
on our folds: trees with three leaves, and a GLM without the extra blocks ours carried. One does not: GPT-2
token keys. Then the public board weighs in, and Part 4's refit is checked on held-out folds.

## 5.1 Three leaves, not four

Part 1 settled on 4-leaf trees from fold 0, where 3 leaves scored 0.00003 lower. On all five folds, with
three seeds each (model seed and the encoding's inner folds, as in Part 4), 3 leaves win:

| LightGBM, leading-digit features | seed 42 / 1 / 2 | mean of the seeds | three-seed bag |
|---|---|---|---|
| 2 leaves | 0.95020 | 0.95020 |  |
| 3 leaves | 0.95064 / 0.95032 / 0.95033 | 0.95043 | **0.95063** |
| 4 leaves | 0.95040 / 0.95020 / 0.95023 | 0.95028 | **0.95051** |

Seed 42 happened to be the lucky seed again (0.95064); the honest gain is the one on the seed mean,
+0.00015. One fold is not enough to choose between settings this close: the same happened with column
sampling, where fold 0 alone said −0.00026 and five folds say ±0.00005. One seed of the 3-leaf model:

In [ ]:
run_tree("lgb_diglead_l3", "lgb", XLtr, XLte, LEAD_TE, num_leaves=3)
saved = np.load(find("lgb_diglead_l3_oof.npy", ART))[:len(tr)]
print(f"lgb_diglead_l3 OOF: largest difference to our dataset {np.abs(OOF['lgb_diglead_l3'] - saved).max():.1e}")

## 5.2 A plain GLM is the strongest GLM

Since Part 1 our GLMs have had cubic splines of every numeric column and one-hot columns for the values of
the numeric columns with up to 1000 distinct values, next to the target-encoding logits. A public notebook
fitted the logits with nothing else and scored higher. Adding our blocks back one at a time (fold 0,
`C=0.3`, every value and digit key encoded):

| GLM, fold 0 | AUC | change |
|---|---|---|
| TE logits + raw columns (plain) | 0.94962 | |
| + TE smoothing 50 instead of 20 | 0.94976 | +1.4 × 1e-4 |
| + splines | 0.94932 | -3.0 × 1e-4 |
| + one-hot of values (≤ 1000 distinct) | 0.94796 | -16.6 × 1e-4 |
| + both (the Part 1-3 GLM design) | 0.94776 | -18.6 × 1e-4 |
| + GPT-2 token keys | 0.94951 | -1.1 × 1e-4 |
| + one-hot values and keys, C 0.01 (Part 4) | 0.94937 | -2.5 × 1e-4 |

The encodings already give each column its shape, value by value. The value one-hots add about 1,500 free
coefficients that re-learn the same shape from fewer rows each and cost 0.0017; the splines cost a little
too. Part 4's joint one-hot GLM was probably in part repairing that damage with its strong penalty: on top of the plain
design the same one-hots cost 0.0002 to 0.0026. The penalty itself hardly matters for the plain GLM
(`C` 0.1 / 0.3 / 1.0: 0.94962 / 0.94962 / 0.94962); smoothing the encodings more (50 instead of 20) helps a
little. Five folds:

| GLM | 5-fold CV AUC |
|---|---|
| Part 3: splines + one-hot + TE logits (`glm_dig`) | 0.94879 |
| Part 4: + joint one-hot of values and keys (`glm_ohk_c01`) | 0.94991 |
| Part 5: plain, TE smoothing 20 (`glm_dl`) | 0.95041 |
| Part 5: plain, TE smoothing 50, 3 seeds (`glm_dl_s50_bag`) | 0.95051 |

It trains in a few minutes, so it runs here:

In [ ]:
def plain_design():
    """Raw numeric columns (log for the skewed ones), standardized; the flags; one-hot categoricals. Nothing else."""
    num = full[NUMS].astype(float).copy()
    for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"):
        num[c] = np.log1p(num[c])
    return sparse.hstack([sparse.csr_matrix(StandardScaler().fit_transform(num)),
                          sparse.csr_matrix(full[FLAGS].values.astype(float)),
                          OneHotEncoder(handle_unknown="ignore").fit_transform(full[CATS].astype(str))]).tocsr()


run_glm("glm_dl_s50", max_card=0, C=0.3, with_te=True, src=(XLtr, XLte), te_cols=LEAD_TE, base=plain_design(), smooth=50.0)
saved = np.load(find("glm_dl_s50_oof.npy", ART))[:len(tr)]
print(f"glm_dl_s50 OOF: largest difference to our dataset {np.abs(OOF['glm_dl_s50'] - saved).max():.1e}")
print(f"rank correlation with Part 4's GLM: {spearmanr(OOF['glm_dl_s50'], OOF['glm_ohk_c01']).correlation:.4f}, "
      f"with the 3-leaf LightGBM: {spearmanr(OOF['glm_dl_s50'], OOF['lgb_diglead_l3']).correlation:.4f}")

## 5.3 GPT-2 token keys: no gain

If the generator is a language model, it writes `1269.35` as the GPT-2 tokens `' 12' '69' '.' '35'`, not
digit by digit. Keys built from those tokens (first, second and last token, the number of tokens, first
token with the count, the integer-part tokens) for all six float columns, target-encoded like the digit keys,
read from the CSV text:

```python
from tokenizers import Tokenizer
gpt2 = Tokenizer.from_pretrained("gpt2")
tokens = {v: gpt2.encode(" " + v).tokens for v in csv_text[col].unique()}   # ' 1269.35' -> ['Ġ12', '69', '.', '35']
```

| GLM, fold 0 | without | with the token keys |
|---|---|---|
| plain GLM | 0.94962 | 0.94951 |
| GLM with splines + one-hot | 0.94776 | 0.94761 |

The token keys cut the numbers at other places than the digit keys, but they add nothing the digit keys do
not already carry (Part 4 found the same for the trees). The plain GLM's lead over ours came from the design,
not from the tokens.

## 5.4 Refit or fold average, on held-out folds

Part 4 replaced every test prediction by one refit on all training rows because one public-board comparison
said +0.00019. Public notebooks measured the opposite for deeper XGBoost. So here is the comparison on our
own data, one level down: each of three folds is held out as a "test set", and the other four folds play the
training set. On them, the 3-leaf recipe is trained exactly as for the real test set (four fold models,
early stopping, three seeds) and scored on the held-out fold:

| held out | fold average | refit on all four folds | rank mean of both |
|---|---|---|---|
| fold 0 | 0.94972 | 0.94973 (+0.2) | 0.94981 (+0.9) |
| fold 1 | 0.95125 | 0.95125 (+0.0) | 0.95134 (+0.8) |
| fold 2 | 0.95574 | 0.95556 (-1.8) | 0.95572 (-0.2) |

(AUC, changes vs the fold average × 1e-4.) Single models: the refit loses to the four-model average in 7 of
9 seeds. As three-seed bags it is no better on average (−0.00005), while the rank mean of the two is best
on average (+0.00005). The public board disagrees once more: the CV-best blend scored 0.95132 with refits and
0.95119 with the rank mean. The differences are small either way; the code is below (`RUN_REFIT_CHECK`).

In [ ]:
RUN_REFIT_CHECK = False      # True: 3 held-out folds x 3 seeds x 5 LightGBM fits (an hour or more on 4 cores)


def refit_check(k, seeds=((SEED, 0), (1, 1), (2, 2)), **over):
    """Fold k as the test set, the other four folds as the training set; returns AUC of (a), (b), (c)."""
    hold, inner = FOLDS[k][1], [FOLDS[j][1] for j in range(N_FOLDS) if j != k]
    pool, res = np.concatenate(inner), []
    for s, ts in seeds:
        avg, its = np.zeros(len(hold)), []
        for j, b in enumerate(inner):
            a = np.concatenate([inner[i] for i in range(len(inner)) if i != j])
            Xa, (Xb, Xh) = add_te(XLtr.iloc[a], y[a], [XLtr.iloc[b], XLtr.iloc[hold]], LEAD_TE, seed=ts)
            m = lgb.LGBMClassifier(**{**LGB, **over}, random_state=s)
            m.fit(drop_keys(Xa), y[a], eval_set=[(drop_keys(Xb), y[b])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(300, verbose=False)])
            avg += m.predict_proba(drop_keys(Xh))[:, 1] / len(inner)
            its.append(m.best_iteration_)
        Xa, (Xh,) = add_te(XLtr.iloc[pool], y[pool], [XLtr.iloc[hold]], LEAD_TE, seed=ts)
        m = lgb.LGBMClassifier(**{**LGB, **over, "n_estimators": int(round(np.mean(its) * 1.25))}, random_state=s)
        res.append((avg, m.fit(drop_keys(Xa), y[pool]).predict_proba(drop_keys(Xh))[:, 1]))
    a, b = np.mean([r[0] for r in res], axis=0), np.mean([r[1] for r in res], axis=0)
    return [roc_auc_score(y[hold], v) for v in (a, b, (rankdata(a) + rankdata(b)) / 2)]


if RUN_REFIT_CHECK:
    for k in range(3):
        print(f"held-out fold {k}: fold average, refit, rank mean =", np.round(refit_check(k, num_leaves=3), 6))

## 5.5 What the public board said

Six submissions after Part 4. "Expected" is the CV difference to Part 4 and the standard deviation of that
difference over random 60,000-row subsets of our out-of-fold rows (the size of the public board):

| submission | 5-fold CV | expected vs Part 4 | public LB (Part 4: 0.95154) |
|---|---|---|---|
| CV best: 3-leaf bag, plain GLM 3/9, EBM i25, ... (refit) | 0.95103 | +0.00024 ± 0.00016 | 0.95132 |
| same, test = rank mean of fold average and refit | 0.95103 | +0.00024 ± 0.00016 | 0.95119 |
| Part 4 with the 3-leaf bag | 0.95082 | +0.00003 ± 0.00006 | 0.95154 |
| Part 4 with the plain GLM | 0.95091 | +0.00013 ± 0.00010 | 0.95136 |
| Part 4 with half of its GLM weight on the plain GLM | 0.95086 | +0.00008 ± 0.00005 | 0.95147 |
| **CV best without the plain GLM (this notebook)** | 0.95092 | +0.00014 ± 0.00010 | **0.95160** |

The 3-leaf swap landed where expected. The GLM swap landed about three standard deviations below its
expectation, and so did the CV-best blend (which gives the plain GLM 3/9). Half the GLM weight cost about
half as much; that is consistent, but not independent evidence (the same public rows score the same GLM
predictions). Without the plain GLM, the best-CV blend landed within its expected range, at 0.95160. On 60,000 public rows that is unlikely
to be chance alone, but Part 4's 0.95154 was also the best of several submissions, so it is probably a
little lucky itself. We looked for something in the test rows that the plain GLM could be reacting to and
found nothing: its test predictions correlate with the other members exactly as its out-of-fold
predictions do (0.989 vs 0.986 with the 4-leaf bag), its logits are spread the same on training, validation
and test rows, and its largest coefficient is 0.7.

So two readings remain: the public 20% is noise and the CV (700,000 rows) is right, or the plain GLM is
worse on the test rows for a reason we have not found. The private board (the other 80%) decides. Kaggle
lets us pick two final submissions; ours will be one of each: the CV-best blend and the blend both
sides agree on (this notebook's `submission.csv`).

## 5.6 The Part 5 blend

The best-CV blend without the plain GLM: the 3-leaf bag; the joint GLM of Part 4 with more key levels
(one-hots also of `int(x/100)`, the first digit, the ones digit and both decimals; the leading digits
encoded; TE smoothing 50, `C=0.005`; two encoding seeds); the EBM with the leading digits and 25
interactions (four seeds); the smoothing-100 LightGBM with five seeds; Part 3's raw-view and 7-leaf trees.
Weights by rank hill climbing on the out-of-fold predictions, test predictions from the refits as in
Part 4: the notebook's `submission.csv`. The CV-best blend (with the plain GLM) goes to
`submission_cv_best.csv`. The bags are loaded from our dataset; `RETRAIN_P5 = True` retrains the 3-leaf bag
with Part 4's `bag_trees` (the other three are variants of the Part 3 and Part 4 code).

In [ ]:
RETRAIN_P5 = False
P5_W = {'lgb_diglead_l3_bag': 2, 'glm_ohk2_bag': 2, 'ebm_i25_bag4': 2, 'lgb_dig_s100_bag5': 1, 'lgb_rawdig_l4': 1, 'lgb_dig_l7': 1}
CV_BEST_W = {'lgb_diglead_l3_bag': 1, 'glm_dl_s50_bag': 3, 'lgb_dig_s100_bag5': 1, 'ebm_i25_bag4': 2, 'lgb_rawdig_l4': 1, 'lgb_dig_l7': 1}


def rank_blend(weights, side):
    w = np.array(list(weights.values()), dtype=float)
    return np.column_stack([rankdata(side[n]) / len(side[n]) for n in weights]) @ (w / w.sum())


for name in ("lgb_diglead_l3_bag", "glm_dl_s50_bag", "glm_ohk2_bag", "lgb_dig_s100_bag5", "ebm_i25_bag4"):
    if RETRAIN_P5 and name == "lgb_diglead_l3_bag":
        OOF[name], FULL[name] = bag_trees("lgb_diglead_l3", "lgb", XLtr, XLte, LEAD_TE, num_leaves=3)
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", ART))[:len(tr)].astype(float)
        FULL[name] = np.load(find(f"{name}_full.npy", ART))[:len(te)].astype(float)

for label, W in (("Part 5 blend", P5_W), ("CV-best blend", CV_BEST_W)):
    print(f"{label}: OOF AUC {roc_auc_score(y, rank_blend(W, OOF)):.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P5_W],
                      "weight": [f"{v}/9" for v in P5_W.values()]}, index=list(P5_W)).round(5))

pd.DataFrame({"id": te["id"], TARGET: rank_blend(CV_BEST_W, FULL)}).to_csv("submission_cv_best.csv", index=False)
sub = pd.DataFrame({"id": te["id"], TARGET: rank_blend(P5_W, FULL)})
sub.to_csv("submission.csv", index=False)
sub.head()

## 5.7 What did not help in Part 5

| idea | result | note |
|---|---|---|
| GPT-2 token keys (first, second, last token, count, …) of all six float columns | −0.00011 (GLM, fold 0) | idea from bulsahkecici; also no gain in our trees (Part 4) |
| Splines and one-hot values in the GLM next to the TE logits | −0.0003 / −0.0017 (fold 0) | the encodings already carry the shape |
| Joint one-hot keys (Part 4) on top of the plain GLM | −0.0002 to −0.0026 (fold 0) | C 0.01 / 0.03 / 0.1 |
| 2 leaves | 0.95021 (5-fold) | 3 leaves 0.95064, 4 leaves 0.95040 (seed 42) |
| GLM: TE smoothing 5 / 100 / 200 | −0.0003 / +0.00015 / +0.00004 (fold 0) | 50 kept (+0.00014) |
| GLM: more digit keys (ones and tens digit, int(x/100)) | −0.00005 / 0.0000 (fold 0) |  |
| LightGBM column sampling 1.0 instead of 0.5 | ±0.00005 (5-fold) | digit-key gain the same with and without (asked in the comments) |

---
## Next

- Find out why the plain GLM wins on 700k out-of-fold rows and loses on the public 20%.
- Pick the two final submissions so that one follows the CV and one the public board.

This notebook will keep growing. If it was useful, an upvote helps the series continue;
questions and corrections are very welcome in the comments.